In [1]:
# Cell 1 — Install (run first, before anything else)
!pip install mediapipe protobuf==5.29.4 opencv-python-headless xgboost shap scipy pyyaml openpyxl -q

import mediapipe as mp
print('mediapipe:', mp.__version__)
print('tasks:', hasattr(mp, 'tasks'))
print('✓ Ready')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.7/319.7 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 112.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 14.6 MB/s eta 0:00:00
mediapipe: 0.10.35
tasks: True
✓ Ready


In [8]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 2 — paste this into your Colab Cell 2 (replaces existing Cell 2)
#
# Changes from your old Cell 2:
#   - HARDWARE_MODES reduced from 5 → 2 (assist / resist)
#   - N_MODES = 2
#   - RehabNet._N_MODES = 2 baked in (prevents IndexError on embedding lookup)
#   - BiLSTM added to RehabNet (bilstm + bilstm_proj layers)
#   - ex_head added (exercise classification head)
#   - shared layer input: 290 (was 162) to match new concat
#   - generate_feedback() added (rule-based clinical messages)
#   - run_loso() now returns (results, best_model) and saves best model to Drive
# ═══════════════════════════════════════════════════════════════════════════

import os
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from scipy.signal import savgol_filter
from scipy.interpolate import interp1d
from typing import List, Optional
from collections import Counter

TARGET_LEN  = 150
N_JOINTS    = 6
IN_CHANNELS = 3
MIN_FRAMES  = 20
FPS         = 30
N_EPOCHS    = 60

UIPRMD_CSV = '/content/drive/MyDrive/datasets/UIPRMD_timeseries.csv'
KIMORE_CSV = '/content/drive/MyDrive/datasets/squats_tabular_timeseries_binary.csv'
KERAAL_CSV = '/content/drive/MyDrive/datasets/keraal_final_binary.csv'

JOINT_ORDER = ['l_hip','l_knee','l_ankle','r_hip','r_knee','r_ankle']

EXERCISE_LIST = [
    'deep_squat','hurdle_step','inline_lunge','side_lunge',
    'sit_to_stand','straight_leg_raise','squat','ctk_squat','unknown'
]
EXERCISE2IDX = {e: i for i, e in enumerate(EXERCISE_LIST)}
N_EXERCISES  = len(EXERCISE_LIST)   # 9

# ── 2 hardware modes (was 5) ──────────────────────────────────────────────────
# mode 0 = assist : device helps patient (post-surgery / injured)
#                   torque decreases as quality improves (less help needed)
# mode 1 = resist : device resists patient (healthy rehab)
#                   torque increases as quality improves (more challenge)
HARDWARE_MODES = {
    0: {'name': 'assist', 'scale': -0.3, 'bias': 0.5, 'min': 0.0, 'max': 0.8},
    1: {'name': 'resist', 'scale':  0.6, 'bias': 0.1, 'min': 0.0, 'max': 0.8},
}
N_MODES = len(HARDWARE_MODES)   # 2

def compute_torque(quality: float, mode: int,
                   velocity_proxy: float = 0.0) -> float:
    m = HARDWARE_MODES.get(mode, HARDWARE_MODES[0])
    return float(np.clip(m['scale'] * quality + m['bias'], m['min'], m['max']))

# ── feedback messages ─────────────────────────────────────────────────────────
FEEDBACK_MAP = {
    'insufficient_ROM':   'Try to bend your knee a little further.',
    'knee_valgus':        '⚠ Keep your knee aligned over your toes — don\'t let it cave inward.',
    'asymmetric':         'Try to put equal weight on both legs.',
    'trunk_compensation': 'Keep your back straight — try not to lean forward.',
    'too_fast':           '⚠ Slow down — take about 3 seconds for each bend.',
    'too_slow':           'Good control — you can move a little more smoothly.',
    'high_jerk':          'Try to move more smoothly — avoid sudden jerky movements.',
}

def generate_feedback(label: int, quality: float,
                      scalars: np.ndarray) -> str:
    """
    Rule-based feedback from 10-dim scalar vector.
    scalars: [l_rom/180, r_rom/180, l_peak/180, r_peak/180,
              sym/100, vel/200, jerk/10, trunk, lag/150, smooth]
    """
    if label == 1 and quality >= 0.75:
        return 'Great rep — keep it up!'
    if label == 1 and quality >= 0.5:
        return 'Good rep — maintain that form.'
    rom_l  = float(scalars[0]) * 180
    rom_r  = float(scalars[1]) * 180
    sym    = float(scalars[4]) * 100
    vel    = float(scalars[5]) * 200
    jerk   = float(scalars[6]) * 10
    trunk  = float(scalars[7])
    smooth = float(scalars[9])
    if vel > 80:              return FEEDBACK_MAP['too_fast']
    if min(rom_l, rom_r) < 50: return FEEDBACK_MAP['insufficient_ROM']
    if trunk > 0.35:          return FEEDBACK_MAP['trunk_compensation']
    if sym > 25:              return FEEDBACK_MAP['asymmetric']
    if jerk > 8:              return FEEDBACK_MAP['high_jerk']
    if vel < 5 and smooth < 0.2: return FEEDBACK_MAP['too_slow']
    return 'Focus on smooth, controlled movement.'

EPS = 1e-6

def safe_np(x, fill=0.0, clip=None, dtype=np.float32):
    x = np.asarray(x, dtype=dtype)
    x = np.nan_to_num(x, nan=fill, posinf=fill, neginf=fill)
    if clip is not None:
        x = np.clip(x, -clip, clip)
    return x.astype(dtype)

def valid_np(x) -> bool:
    x = np.asarray(x)
    return x.size > 0 and np.isfinite(x).all()

def build_adjacency() -> np.ndarray:
    A = np.zeros((N_JOINTS, N_JOINTS), dtype=np.float32)
    for i, j in [(0,1),(1,2),(3,4),(4,5),(0,3),(1,4),(2,5)]:
        A[i,j] = A[j,i] = 1.0
    np.fill_diagonal(A, 1.0)
    d = np.diag(1.0 / np.sqrt(A.sum(axis=1) + EPS))
    return (d @ A @ d).astype(np.float32)

ADJ = build_adjacency()

def _smooth(arr, win=7, poly=2):
    arr = safe_np(arr)
    if len(arr) < win + 2: return arr
    w = win if win % 2 == 1 else win + 1
    w = max(w, poly+2 if (poly+2)%2==1 else poly+3)
    try:   return safe_np(savgol_filter(arr, w, poly))
    except: return arr

def _resample(arr, n):
    arr = safe_np(arr)
    if len(arr) == n:  return arr
    if len(arr) < 2:   return np.zeros(n, dtype=np.float32)
    return safe_np(np.interp(np.linspace(0,1,n), np.linspace(0,1,len(arr)), arr))

def _fill_missing(kps):
    kps = safe_np(kps); T, J, C = kps.shape
    for j in range(J):
        for c in range(C):
            col = kps[:,j,c]
            z   = (~np.isfinite(col)) | (col == 0.0)
            if z.all():
                opp = j+3 if j < 3 else j-3
                kps[:,j,c] = kps[:,opp,c]
            elif z.any() and (~z).sum() >= 2:
                f = interp1d(np.where(~z)[0], col[~z],
                             bounds_error=False, fill_value='extrapolate')
                kps[:,j,c] = f(np.arange(T))
    return safe_np(kps)

def _normalise(kps):
    kps = safe_np(kps)
    hip = ((kps[:,0,:] + kps[:,3,:]) / 2.0).mean(axis=0)
    kps = kps - hip
    l   = np.mean(np.linalg.norm(kps[:,0,:] - kps[:,2,:], axis=1))
    r   = np.mean(np.linalg.norm(kps[:,3,:] - kps[:,5,:], axis=1))
    scale = (l + r) / 2.0
    if not np.isfinite(scale) or scale < 1e-4: scale = 1.0
    return safe_np(np.clip(kps / scale, -5.0, 5.0), clip=5.0)

def process_keypoints(raw: np.ndarray) -> np.ndarray:
    raw = safe_np(raw)
    if raw.ndim != 3 or raw.shape[1] != N_JOINTS or raw.shape[2] != 3:
        raise ValueError(f'Expected (T,{N_JOINTS},3), got {raw.shape}')
    if len(raw) < MIN_FRAMES:
        raise ValueError(f'Too short: {len(raw)} frames')
    for j in range(N_JOINTS):
        for c in range(3): raw[:,j,c] = _smooth(raw[:,j,c])
    raw = _fill_missing(raw)
    out = np.zeros((TARGET_LEN, N_JOINTS, 3), dtype=np.float32)
    for j in range(N_JOINTS):
        for c in range(3): out[:,j,c] = _resample(raw[:,j,c], TARGET_LEN)
    return safe_np(_normalise(out), clip=5.0)

def extract_scalars(kps: np.ndarray) -> np.ndarray:
    kps = safe_np(kps, clip=5.0)
    def _ang(a, b, c):
        ba, bc = a-b, c-b
        d = np.maximum(np.linalg.norm(ba,1) * np.linalg.norm(bc,1), EPS)
        return np.degrees(np.arccos(np.clip(np.sum(ba*bc,1)/d, -1, 1)))
    lk = _ang(kps[:,0,:], kps[:,1,:], kps[:,2,:])
    rk = _ang(kps[:,3,:], kps[:,4,:], kps[:,5,:])
    lr = float(np.nanmax(lk) - np.nanmin(lk))
    rr = float(np.nanmax(rk) - np.nanmin(rk))
    sym  = abs(lr-rr) / ((lr+rr)/2+EPS) * 100 if (lr+rr) > 1 else 0.0
    vel  = float(np.max(np.abs(np.diff(lk))) * FPS) if len(lk) > 1 else 0.0
    hip  = (kps[:,0,:] + kps[:,3,:]) / 2.0
    jk   = np.diff(np.diff(hip, axis=0), axis=0)
    jerk = float(np.nanmean(np.linalg.norm(jk, axis=1))) if len(jk) else 0.0
    pk   = int(np.argmin(lk))
    trunk= float(abs(hip[pk,0] - hip[0,0]))
    lkc, rkc = lk - np.nanmean(lk), rk - np.nanmean(rk)
    lag  = float(abs(np.argmax(np.correlate(lkc, rkc, 'full')) - (len(lk)-1))) \
           if np.nanstd(lkc) > EPS and np.nanstd(rkc) > EPS else 0.0
    smooth = float(np.clip(1.0 / (np.nanstd(np.diff(lk)) + EPS) / 100, 0, 1))
    return safe_np(np.array([
        np.clip(lr/180,0,1),   np.clip(rr/180,0,1),
        np.clip(np.min(lk)/180,0,1), np.clip(np.min(rk)/180,0,1),
        np.clip(sym/100,0,5),  np.clip(vel/200,0,5),
        np.clip(jerk/10,0,5),  np.clip(trunk,0,5),
        np.clip(lag/TARGET_LEN,0,1), smooth,
    ], dtype=np.float32), fill=0.0, clip=10.0)

def validate_ps1_frame(frame: dict) -> bool:
    for j in ['HipLeft','KneeLeft','AnkleLeft','HipRight','KneeRight','AnkleRight']:
        for ax in ['x','y','z']:
            v = frame.get(f'{j}_{ax}')
            if v is None or np.isnan(float(v)) or abs(float(v)) > 10.0:
                return False
    return True

def ps1_frame_to_keypoint(frame: dict) -> np.ndarray:
    row = np.zeros((N_JOINTS, 3), dtype=np.float32)
    for ji, jname in enumerate(['HipLeft','KneeLeft','AnkleLeft',
                                  'HipRight','KneeRight','AnkleRight']):
        for ci, ax in enumerate(['x','y','z']):
            row[ji, ci] = float(frame.get(f'{jname}_{ax}', 0.0))
    return row

def _kps_from_df_rows(df, xyz_cols):
    return df[xyz_cols].values.astype(np.float32).reshape(len(df), N_JOINTS, 3)

def load_uiprmd() -> List[dict]:
    if not os.path.exists(UIPRMD_CSV):
        print(f'UI-PRMD not found: {UIPRMD_CSV}'); return []
    df       = pd.read_csv(UIPRMD_CSV)
    xyz_cols = [f'{j}_{ax}' for j in JOINT_ORDER for ax in ['x','y','z']]
    samples  = []
    for (subj, mov, ex_id, lv), grp in df.groupby(['subject','movement','exercise_id','label']):
        grp = grp.sort_values('frame')
        raw = _kps_from_df_rows(grp, xyz_cols) / 1000.0
        try:
            kps = process_keypoints(raw)
            ex  = grp['exercise'].iloc[0]
            samples.append({'keypoints': kps, 'adj': ADJ,
                             'label': int(lv), 'quality': float(lv),
                             'exercise': ex,
                             'exercise_idx': EXERCISE2IDX.get(ex, EXERCISE2IDX['unknown']),
                             'subject': f'uiprmd_{subj}', 'source': 'uiprmd',
                             'scalars': extract_scalars(kps)})
        except Exception as e:
            print(f'  skip uiprmd {subj}/{mov}/{ex_id}: {e}')
    print(f'UI-PRMD: {len(samples)} samples')
    return samples

def load_kimore() -> List[dict]:
    if not os.path.exists(KIMORE_CSV):
        print(f'KIMORE not found: {KIMORE_CSV}'); return []
    df = pd.read_csv(KIMORE_CSV)
    xyz_cols = []
    for jn in ['HipLeft','KneeLeft','AnkleLeft','HipRight','KneeRight','AnkleRight']:
        xyz_cols += [f'{jn}_x', f'{jn}_y', f'{jn}_z']
    df['quality_norm'] = df[['cTS','cPO','cCF']].mean(axis=1) / 50.0
    samples = []
    for subj, grp in df.groupby('Subject'):
        grp = grp.sort_values('FrameID')
        raw = _kps_from_df_rows(grp, xyz_cols)
        try:
            kps = process_keypoints(raw)
            samples.append({'keypoints': kps, 'adj': ADJ,
                             'label': int(grp['label'].iloc[0]),
                             'quality': float(grp['quality_norm'].mean()),
                             'exercise': 'squat',
                             'exercise_idx': EXERCISE2IDX['squat'],
                             'subject': f'kimore_{subj}', 'source': 'kimore',
                             'scalars': extract_scalars(kps)})
        except Exception as e:
            print(f'  skip kimore {subj}: {e}')
    print(f'KIMORE: {len(samples)} samples')
    return samples

def load_keraal() -> List[dict]:
    if not os.path.exists(KERAAL_CSV):
        print(f'Keraal not found: {KERAAL_CSV}'); return []
    df = pd.read_csv(KERAAL_CSV)
    xyz_cols = []
    for jn in ['HipLeft','KneeLeft','AnkleLeft','HipRight','KneeRight','AnkleRight']:
        xyz_cols += [f'{jn}_x', f'{jn}_y', f'{jn}_z']
    samples = []
    for ann_key, grp in df.groupby('ann_key'):
        grp = grp.sort_values('FrameID') if 'FrameID' in grp.columns else grp
        raw = _kps_from_df_rows(grp, xyz_cols)
        try:
            kps  = process_keypoints(raw)
            subj = '-'.join(str(ann_key).split('-')[:3])
            samples.append({'keypoints': kps, 'adj': ADJ,
                             'label': int(grp['label'].iloc[0]),
                             'quality': float(grp['label'].iloc[0]),
                             'exercise': 'ctk_squat',
                             'exercise_idx': EXERCISE2IDX['ctk_squat'],
                             'subject': f'keraal_{subj}', 'source': 'keraal',
                             'scalars': extract_scalars(kps)})
        except Exception as e:
            print(f'  skip keraal {ann_key}: {e}')
    print(f'Keraal: {len(samples)} samples')
    return samples

def build_master_dataset() -> List[dict]:
    raw = load_uiprmd() + load_kimore() + load_keraal()
    clean = []
    for s in raw:
        try:
            kps = safe_np(s['keypoints'], clip=5.0)
            sc  = safe_np(s.get('scalars', extract_scalars(kps)), clip=10.0)
            q   = float(np.nan_to_num(s.get('quality', s.get('label', 1)),
                                      nan=float(s.get('label', 1))))
            ex  = int(s.get('exercise_idx', EXERCISE2IDX['unknown']))
            if kps.shape != (TARGET_LEN, N_JOINTS, 3): raise ValueError('bad kps shape')
            if sc.shape  != (10,):                     raise ValueError('bad scalar shape')
            if not valid_np(kps) or not valid_np(sc):  raise ValueError('NaN/Inf')
            if not (0 <= ex < N_EXERCISES): ex = EXERCISE2IDX['unknown']
            s.update({'keypoints': kps, 'scalars': sc,
                      'label': int(s.get('label', 1)),
                      'quality': float(np.clip(q, 0, 1)),
                      'exercise_idx': ex})
            clean.append(s)
        except Exception as e:
            print(f"  DROP {s.get('subject','?')}: {e}")
    print(f'\nMaster: {len(clean)} samples')
    print(f'  correct={sum(s["label"]==1 for s in clean)}'
          f'  incorrect={sum(s["label"]==0 for s in clean)}')
    print(f'  sources:   {set(s["source"]   for s in clean)}')
    print(f'  exercises: {set(s["exercise"] for s in clean)}')
    return clean

def loso_split(samples, held_out):
    return ([s for s in samples if s['subject'] != held_out],
            [s for s in samples if s['subject'] == held_out])

def get_uiprmd_subjects(samples):
    return sorted({s['subject'] for s in samples if s['source'] == 'uiprmd'})

class RehabDataset(Dataset):
    def __init__(self, samples, augment=False):
        self.samples = samples; self.augment = augment
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        s   = self.samples[idx]
        kps = safe_np(s['keypoints'], clip=5.0)
        if self.augment: kps = self._aug(kps)
        return {'keypoints':    torch.from_numpy(safe_np(kps, clip=5.0)).permute(2,0,1).float(),
                'adj':          torch.from_numpy(ADJ).float(),
                'label':        torch.tensor(s['label'],        dtype=torch.long),
                'quality':      torch.tensor(s['quality'],      dtype=torch.float32),
                'scalars':      torch.from_numpy(safe_np(s['scalars'], clip=10.0)).float(),
                'exercise_idx': torch.tensor(s['exercise_idx'], dtype=torch.long),
                'exercise':     s['exercise'], 'subject': s['subject']}
    def _aug(self, kps):
        kps = safe_np(kps, clip=5.0)
        if np.random.rand() < 0.5:
            n = max(int(TARGET_LEN * np.random.uniform(0.8,1.2)), MIN_FRAMES)
            tmp = np.zeros((n, N_JOINTS, 3), dtype=np.float32)
            for j in range(N_JOINTS):
                for c in range(3): tmp[:,j,c] = _resample(kps[:,j,c], n)
            kps = np.zeros((TARGET_LEN, N_JOINTS, 3), dtype=np.float32)
            for j in range(N_JOINTS):
                for c in range(3): kps[:,j,c] = _resample(tmp[:,j,c], TARGET_LEN)
        if np.random.rand() < 0.5:
            kps[:,[0,1,2,3,4,5],:] = kps[:,[3,4,5,0,1,2],:]
            kps[:,:,0] *= -1.0
        if np.random.rand() < 0.5:
            kps += np.random.randn(*kps.shape).astype(np.float32) * 0.02
        if np.random.rand() < 0.3:
            th = np.radians(np.random.uniform(-10,10))
            x, z = kps[:,:,0].copy(), kps[:,:,2].copy()
            kps[:,:,0] = np.cos(th)*x - np.sin(th)*z
            kps[:,:,2] = np.sin(th)*x + np.cos(th)*z
        return safe_np(kps, clip=5.0)

def collate(batch):
    return {k: torch.stack([b[k] for b in batch])
              if isinstance(batch[0][k], torch.Tensor)
              else [b[k] for b in batch]
            for k in batch[0]}

def build_loaders(train_samples, test_samples, batch_size=32):
    ex_counts = Counter(s['exercise'] for s in train_samples)
    total     = len(train_samples)
    weights   = [total / ex_counts[s['exercise']] for s in train_samples]
    sampler   = WeightedRandomSampler(weights, len(train_samples), replacement=True)
    return (DataLoader(RehabDataset(train_samples, True),  batch_size,
                       sampler=sampler, collate_fn=collate, num_workers=2),
            DataLoader(RehabDataset(test_samples,  False), batch_size, False,
                       collate_fn=collate, num_workers=2))

class STGCNBlock(nn.Module):
    def __init__(self, c_in, c_out, ks=9, stride=1):
        super().__init__()
        self.gcn = nn.Linear(c_in, c_out)
        self.tcn = nn.Sequential(
            nn.BatchNorm2d(c_out), nn.ReLU(),
            nn.Conv2d(c_out, c_out, (ks,1), (stride,1), ((ks-1)//2,0)),
            nn.BatchNorm2d(c_out), nn.Dropout(0.1))
        self.res = (nn.Sequential(nn.Conv2d(c_in, c_out, 1, (stride,1)),
                                   nn.BatchNorm2d(c_out))
                    if c_in != c_out or stride != 1 else nn.Identity())
        self.relu = nn.ReLU()
    def forward(self, x, A):
        B, C, T, V = x.shape
        if A.dim() == 3: A = A[0]
        xs = x.permute(0,2,3,1).contiguous().view(B*T, V, C)
        xs = torch.bmm(A.unsqueeze(0).expand(B*T,-1,-1), xs)
        xs = self.gcn(xs).view(B, T, V, -1).permute(0,3,1,2)
        return self.relu(
            torch.nan_to_num(self.tcn(xs), 0.0) +
            torch.nan_to_num(self.res(x),  0.0))

class STGCN(nn.Module):
    def __init__(self):
        super().__init__()
        self.input_norm = nn.LayerNorm(IN_CHANNELS * N_JOINTS)
        self.b1 = STGCNBlock(3, 32)
        self.b2 = STGCNBlock(32, 64)
        self.b3 = STGCNBlock(64, 128)
    def forward(self, x, A):
        B, C, T, V = x.shape
        x  = torch.clamp(x, -5.0, 5.0)
        xf = x.permute(0,2,1,3).contiguous().view(B*T, C*V)
        xf = self.input_norm(xf).view(B, T, C, V).permute(0,2,1,3)
        return self.b3(self.b2(self.b1(xf, A), A), A).mean(3).permute(0,2,1)

class ClinicalTransformer(nn.Module):
    def __init__(self, d=128, heads=4, d_ff=256, drop=0.1):
        super().__init__()
        pe  = torch.zeros(TARGET_LEN+1, d)
        pos = torch.arange(TARGET_LEN+1).unsqueeze(1).float()
        div = torch.exp(torch.arange(0,d,2).float() * (-np.log(10000.0)/d))
        pe[:,0::2] = torch.sin(pos*div)
        pe[:,1::2] = torch.cos(pos*div)
        self.register_buffer('pe', pe.unsqueeze(0))
        self.cls  = nn.Parameter(torch.randn(1,1,d) * 0.02)
        self.drop = nn.Dropout(drop)
        mk = lambda: nn.TransformerEncoderLayer(
            d, heads, d_ff, drop, batch_first=True, norm_first=True)
        self.l1, self.l2 = mk(), mk()
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
    def forward(self, x, return_attn=False):
        B, T, _ = x.shape
        x = torch.cat([self.cls.expand(B,-1,-1), x], 1)
        x = self.drop(x + self.pe[:, :T+1, :])
        x = self.n1(self.l1(x))
        if return_attn:
            sa  = self.l2.self_attn; xn = self.l2.norm1(x)
            out, attn = sa(xn, xn, xn, need_weights=True, average_attn_weights=True)
            x2  = x + self.l2.dropout1(out); xn2 = self.l2.norm2(x2)
            ff  = self.l2.linear2(self.l2.dropout(
                      self.l2.activation(self.l2.linear1(xn2))))
            return self.n2(x2 + self.l2.dropout2(ff))[:,0,:], attn
        return self.n2(self.l2(x))[:,0,:], None

class RehabNet(nn.Module):
    # baked-in constants prevent stale-state IndexError on embedding lookup
    _N_MODES     = 2   # assist / resist
    _N_EXERCISES = 9

    def __init__(self):
        super().__init__()
        self.stgcn       = STGCN()
        self.trans       = ClinicalTransformer()
        self.bilstm      = nn.LSTM(128, 64, num_layers=2, batch_first=True,
                                   bidirectional=True, dropout=0.1)
        self.bilstm_proj = nn.Sequential(
            nn.Linear(128, 128), nn.LayerNorm(128), nn.ReLU(), nn.Dropout(0.1))
        self.mode_emb    = nn.Embedding(self._N_MODES,     16)
        self.ex_emb      = nn.Embedding(self._N_EXERCISES,  8)
        # 128 (cls) + 128 (bilstm) + 10 (scalars) + 16 (mode) + 8 (ex) = 290
        self.shared      = nn.Sequential(
            nn.Linear(290, 128), nn.LayerNorm(128), nn.ReLU(), nn.Dropout(0.2))
        self.classify_head = nn.Linear(128, 2)
        self.quality_head  = nn.Linear(128, 1)
        self.ex_head       = nn.Linear(128, self._N_EXERCISES)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def encode(self, kps, adj, return_attn=False):
        if adj.dim() == 3: adj = adj[0]
        feats = self.stgcn(kps, adj)
        cls, attn = self.trans(feats, return_attn)
        lstm_out, _ = self.bilstm(feats)
        bilstm_ctx  = self.bilstm_proj(lstm_out.mean(1))
        return cls, bilstm_ctx, attn

    def forward(self, kps, adj, scalars, mode=None, exercise_idx=None):
        B, device = kps.shape[0], kps.device
        if mode         is None: mode         = torch.zeros(B, dtype=torch.long, device=device)
        if exercise_idx is None: exercise_idx = torch.zeros(B, dtype=torch.long, device=device)
        mode         = torch.clamp(mode,         0, self._N_MODES     - 1)
        exercise_idx = torch.clamp(exercise_idx, 0, self._N_EXERCISES - 1)
        cls, bilstm_ctx, _ = self.encode(kps, adj)
        x = self.shared(torch.cat([cls, bilstm_ctx, scalars,
                                   self.mode_emb(mode),
                                   self.ex_emb(exercise_idx)], 1))
        return self.classify_head(x), self.quality_head(x).squeeze(1), self.ex_head(cls)

    def predict_exercise(self, kps, adj):
        self.eval()
        with torch.no_grad():
            if adj.dim() == 3: adj = adj[0]
            cls, _, _ = self.encode(kps, adj)
            idx = int(self.ex_head(cls).argmax(1).item())
        return EXERCISE_LIST[idx], idx

def compute_class_weights(train_samples, device):
    n0  = sum(s['label'] == 0 for s in train_samples)
    n1  = sum(s['label'] == 1 for s in train_samples)
    tot = n0 + n1
    return torch.tensor([tot/(2*n0+EPS), tot/(2*n1+EPS)],
                        dtype=torch.float32, device=device)

def _train_epoch(model, loader, opt, device, cw, epoch=0):
    model.train(); A = torch.from_numpy(ADJ).to(device)
    total, used, skipped = 0.0, 0, 0
    for b in loader:
        kps = b['keypoints'].to(device); lbl = b['label'].to(device)
        qual= b['quality'].to(device);   sc  = b['scalars'].to(device)
        ex_idx = b['exercise_idx'].to(device)
        good = (torch.isfinite(kps).flatten(1).all(1) &
                torch.isfinite(sc).all(1) & torch.isfinite(qual))
        if not good.all():
            kps,lbl,qual,sc,ex_idx = (t[good] for t in (kps,lbl,qual,sc,ex_idx))
        if kps.shape[0] == 0: skipped += 1; continue
        mode = torch.zeros(kps.shape[0], dtype=torch.long, device=device)
        lg, qp, ex_lg = model(kps, A, sc, mode, ex_idx)
        if not (torch.isfinite(lg).all() and torch.isfinite(qp).all()):
            skipped += 1; continue
        loss = (F.cross_entropy(lg, lbl, weight=cw) +
                0.3 * F.mse_loss(torch.sigmoid(qp), torch.clamp(qual,0,1)) +
                0.5 * F.cross_entropy(ex_lg, ex_idx))
        if not torch.isfinite(loss): skipped += 1; continue
        opt.zero_grad(set_to_none=True); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        opt.step(); total += loss.item(); used += 1
    if skipped: print(f'  ep{epoch}: skipped {skipped} batches')
    return total / max(used, 1)

@torch.no_grad()
def _evaluate(model, loader, device):
    model.eval(); A = torch.from_numpy(ADJ).to(device)
    preds, labels, quals, qpreds = [], [], [], []
    for b in loader:
        kps = b['keypoints'].to(device); sc  = b['scalars'].to(device)
        ex_idx = b['exercise_idx'].to(device)
        qual   = b['quality'].to(device); lbl = b['label'].to(device)
        good   = torch.isfinite(kps).flatten(1).all(1) & torch.isfinite(sc).all(1)
        if good.sum() == 0: continue
        kps,sc,ex_idx,qual,lbl = (t[good] for t in (kps,sc,ex_idx,qual,lbl))
        mode = torch.zeros(kps.shape[0], dtype=torch.long, device=device)
        lg, qp, _ = model(kps, A, sc, mode, ex_idx)
        preds.extend(lg.argmax(1).cpu().numpy())
        labels.extend(lbl.cpu().numpy())
        quals.extend(qual.cpu().numpy())
        qpreds.extend(torch.sigmoid(qp).cpu().numpy())
    from sklearn.metrics import accuracy_score, f1_score, classification_report
    from scipy.stats import pearsonr
    if not labels: return {'acc': 0.0, 'f1': 0.0, 'quality_r': 0.0}
    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average='macro', zero_division=0)
    r   = pearsonr(quals, qpreds)[0] if len(set(quals)) > 1 else 0.0
    print(classification_report(labels, preds,
          target_names=['incorrect','correct'], zero_division=0))
    print(f'  quality Pearson r={r:.3f}')
    return {'acc': acc, 'f1': f1, 'quality_r': r}

def run_loso(n_epochs=N_EPOCHS, batch_size=32, lr=3e-4, patience=12):
    device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device: {device}')
    samples  = build_master_dataset()
    subjects = get_uiprmd_subjects(samples)
    results  = []; best_overall_model = None; best_overall_f1 = 0.0

    for subj in subjects:
        print(f"\n{'='*55}\nLOSO held-out: {subj}\n{'='*55}")
        tr, te = loso_split(samples, subj)
        tl, vl = build_loaders(tr, te, batch_size)
        cw     = compute_class_weights(tr, device)
        print(f'  class weights: incorrect={cw[0]:.3f}  correct={cw[1]:.3f}')

        model = RehabNet().to(device)
        opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        def lr_fn(ep):
            w = 5
            if ep < w: return (ep+1) / w
            return 0.5 * (1 + np.cos(np.pi * (ep-w) / max(n_epochs-w, 1)))
        sch = torch.optim.lr_scheduler.LambdaLR(opt, lr_fn)

        best_f1 = 0.; best_state = None; no_imp = 0
        for ep in range(n_epochs):
            loss = _train_epoch(model, tl, opt, device, cw, epoch=ep)
            sch.step()
            if (ep+1) % 5 == 0:
                model.eval(); pl, ll = [], []
                with torch.no_grad():
                    A2 = torch.from_numpy(ADJ).to(device)
                    for b in vl:
                        kps = b['keypoints'].to(device)
                        sc  = b['scalars'].to(device)
                        ex_idx = b['exercise_idx'].to(device)
                        lbl    = b['label']
                        good   = (torch.isfinite(kps).flatten(1).all(1) &
                                  torch.isfinite(sc).all(1))
                        if good.sum() == 0: continue
                        kps, sc, ex_idx = kps[good], sc[good], ex_idx[good]
                        lbl = lbl[good.cpu()]
                        mode = torch.zeros(kps.shape[0], dtype=torch.long, device=device)
                        lg, _, _ = model(kps, A2, sc, mode, ex_idx)
                        pl.extend(lg.argmax(1).cpu().numpy())
                        ll.extend(lbl.numpy())
                model.train()
                from sklearn.metrics import f1_score as _f1
                vf1 = _f1(ll, pl, average='macro', zero_division=0)
                print(f'  ep{ep+1:3d} loss={loss:.4f} '
                      f'lr={sch.get_last_lr()[0]:.2e} val_f1={vf1:.3f}')
                if vf1 > best_f1:
                    best_f1 = vf1
                    best_state = copy.deepcopy(model.state_dict())
                    no_imp = 0
                else:
                    no_imp += 1
                    if no_imp >= patience:
                        print(f'  Early stop ep{ep+1}'); break

        if best_state:
            model.load_state_dict(best_state)
        print(f'\n── Eval: {subj} (best val_f1={best_f1:.3f}) ──')
        r = _evaluate(model, vl, device); results.append(r)
        if r['f1'] > best_overall_f1:
            best_overall_f1 = r['f1']
            best_overall_model = copy.deepcopy(model)

    accs = [r['acc'] for r in results]; f1s = [r['f1'] for r in results]
    print(f"\n{'='*55}\nLOSO RESULTS\n"
          f"  Acc={np.mean(accs):.3f}±{np.std(accs):.3f}  "
          f"F1={np.mean(f1s):.3f}±{np.std(f1s):.3f}\n{'='*55}")

    os.makedirs('/content/drive/MyDrive/models', exist_ok=True)
    if best_overall_model is not None:
        torch.save(best_overall_model.state_dict(),
                   '/content/drive/MyDrive/models/rehabnet_best.pth')
        print('Best model saved → /content/drive/MyDrive/models/rehabnet_best.pth')
    return results, best_overall_model

print(f'Cell 2 loaded ✓  N_MODES={N_MODES}  N_EXERCISES={N_EXERCISES}')
print(f'Hardware modes: {[v["name"] for v in HARDWARE_MODES.values()]}')

Cell 2 loaded ✓  N_MODES=2  N_EXERCISES=9
Hardware modes: ['assist', 'resist']


In [10]:
"""
Exercise Classifier — XGBoost on scalar features
From codecl.ipynb: subject-aware 5-fold CV, engineered features
Replaces the neural exercise head with a more accurate XGBoost classifier
"""

import os, pickle
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

try:
    from xgboost import XGBClassifier
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'xgboost', '-q'])
    from xgboost import XGBClassifier

# ── Feature engineering (from codecl.ipynb) ──────────────────────────────────

def engineer_features(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy()

    # ── A. Bilateral asymmetry ─────────────────────────────────────────────
    for joint in ['hip', 'knee', 'ankle']:
        l_rom, r_rom = X[f'left_{joint}_ROM'], X[f'right_{joint}_ROM']
        denom = ((l_rom + r_rom) / 2).replace(0, np.nan)
        X[f'fe_{joint}_ROM_asym']    = (l_rom - r_rom).abs() / denom
        l_vel, r_vel = X[f'left_{joint}_mean_velocity'], X[f'right_{joint}_mean_velocity']
        denom_v = ((l_vel + r_vel) / 2).replace(0, np.nan)
        X[f'fe_{joint}_vel_asym']    = (l_vel - r_vel).abs() / denom_v
        X[f'fe_{joint}_peak_asym']   = (X[f'left_{joint}_peak_angle'] - X[f'right_{joint}_peak_angle']).abs()
        X[f'fe_{joint}_smooth_diff'] = X[f'left_{joint}_smoothness']  - X[f'right_{joint}_smoothness']

    # ── B. Hip-knee kinematic chain ────────────────────────────────────────
    X['fe_left_hip_knee_ROM_ratio']  = X['left_hip_ROM']  / (X['left_knee_ROM']  + 1e-6)
    X['fe_right_hip_knee_ROM_ratio'] = X['right_hip_ROM'] / (X['right_knee_ROM'] + 1e-6)
    X['fe_hk_coupling_asym']         = X['left_hip_knee_coupling'] - X['right_hip_knee_coupling']

    # ── C. Knee flexion deficit ────────────────────────────────────────────
    X['fe_left_knee_flex_deficit']  = X['left_knee_start_angle']  - X['left_knee_peak_angle']
    X['fe_right_knee_flex_deficit'] = X['right_knee_start_angle'] - X['right_knee_peak_angle']
    X['fe_knee_flex_deficit_asym']  = X['fe_left_knee_flex_deficit'] - X['fe_right_knee_flex_deficit']

    # ── D. Ankle dorsiflexion ──────────────────────────────────────────────
    X['fe_ankle_dors_asym']    = (X['left_ankle_dorsiflexion'] - X['right_ankle_dorsiflexion']).abs()
    X['fe_ankle_ROM_combined'] = X['left_ankle_ROM'] + X['right_ankle_ROM']

    # ── E. Trunk lean ──────────────────────────────────────────────────────
    X['fe_trunk_lean_asym']  = (X['left_trunk_lean_proxy'] - X['right_trunk_lean_proxy']).abs()
    X['fe_trunk_lean_total'] = X['left_trunk_lean_proxy'] + X['right_trunk_lean_proxy']

    # ── F. Movement quality composite ─────────────────────────────────────
    for side in ['left', 'right']:
        for joint in ['hip', 'knee', 'ankle']:
            X[f'fe_{side}_{joint}_quality'] = X[f'{side}_{joint}_smoothness'] / (X[f'{side}_{joint}_EC_ratio'] + 1e-6)

    # ── G. Eccentric control ratio ─────────────────────────────────────────
    for side in ['left', 'right']:
        for joint in ['hip', 'knee']:
            desc = X[f'{side}_{joint}_mean_vel_desc']
            asc  = X[f'{side}_{joint}_mean_vel_asc']
            X[f'fe_{side}_{joint}_ecc_ratio'] = desc / (desc + asc + 1e-6)

    # ── H. Hesitation index ────────────────────────────────────────────────
    X['fe_left_total_hesitations']  = X['left_hip_hesitations']  + X['left_knee_hesitations']  + X['left_ankle_hesitations']
    X['fe_right_total_hesitations'] = X['right_hip_hesitations'] + X['right_knee_hesitations'] + X['right_ankle_hesitations']
    X['fe_hesitation_asym']         = X['fe_left_total_hesitations'] - X['fe_right_total_hesitations']

    # ── I. Temporal coordination lag ──────────────────────────────────────
    X['fe_total_temporal_lag'] = (
        X['hip_temporal_lag_fr'].abs() +
        X['knee_temporal_lag_fr'].abs() +
        X['ankle_temporal_lag_fr'].abs()
    )
    return X



# ── Exercise classifier class ─────────────────────────────────────────────────

class ExerciseClassifier:
    """
    Standalone exercise classifier that wraps the XGBoost from codecl.ipynb.
    Used in run_patient_video() to auto-classify exercise from scalar features.

    Usage:
        clf = ExerciseClassifier()
        clf.train(csv_path='/content/drive/MyDrive/datasets/final_cleaned_data.csv')
        exercise_name = clf.predict(scalar_features_dict)
    """

    IDENTIFIERS = ['rep_id', 'subject_id']
    TARGET      = 'exercise'
    IMU_COLS_PREFIX = 'imu_'
    SLR_COLS = ['left_knee_stability_slr','left_knee_min_during_slr',
                'right_knee_stability_slr','right_knee_min_during_slr']

    def __init__(self):
        self.model       = None
        self.le_target   = LabelEncoder()
        self.feature_cols = None
        self.trained     = False

    def _get_drop_cols(self, df):
        imu_cols = [c for c in df.columns if c.startswith(self.IMU_COLS_PREFIX)]
        return list(set(self.IDENTIFIERS + [self.TARGET] + imu_cols + self.SLR_COLS))

    def train(self, csv_path: str, save_path: str = 'exercise_classifier.pkl'):
        print("Training exercise classifier...")
        df     = pd.read_csv(csv_path, low_memory=False)
        groups = df['subject_id'].values

        drop   = self._get_drop_cols(df)
        y      = self.le_target.fit_transform(df[self.TARGET].values)
        X_base = df.drop(columns=drop).copy()

        # Engineer features
        X_eng = engineer_features(X_base)

        # Handle nulls
        null_cols = X_eng.columns[X_eng.isnull().any()].tolist()
        for col in null_cols:
            X_eng[col] = X_eng[col].fillna(X_eng[col].median())

        self.feature_cols = X_eng.columns.tolist()
        X = X_eng.copy()

        # Sample weights for class balance
        sample_weights = compute_sample_weight(class_weight='balanced', y=y)

        # 5-fold subject-aware CV for reporting
        sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
        fold_results = []
        all_y_true, all_y_pred = [], []

        print("Running 5-fold subject-aware CV...")
        for fold, (tr_idx, te_idx) in enumerate(sgkf.split(X.values, y, groups)):
            m = XGBClassifier(
                n_estimators=300, max_depth=5, learning_rate=0.05,
                subsample=0.80, colsample_bytree=0.70, min_child_weight=5,
                gamma=1.0, reg_alpha=0.10, reg_lambda=1.0,
                objective='multi:softprob', num_class=len(self.le_target.classes_),
                eval_metric='mlogloss', random_state=42, n_jobs=-1,
            )
            m.fit(X.values[tr_idx], y[tr_idx],
                  sample_weight=sample_weights[tr_idx],
                  eval_set=[(X.values[te_idx], y[te_idx])],
                  verbose=False)
            pred = m.predict(X.values[te_idx])
            acc  = accuracy_score(y[te_idx], pred)
            f1   = f1_score(y[te_idx], pred, average='macro', zero_division=0)
            fold_results.append({'fold': fold+1, 'acc': acc, 'f1': f1})
            all_y_true.extend(y[te_idx])
            all_y_pred.extend(pred)
            print(f"  Fold {fold+1}: acc={acc:.3f}  f1={f1:.3f}")

        mean_acc = np.mean([r['acc'] for r in fold_results])
        mean_f1  = np.mean([r['f1']  for r in fold_results])
        print(f"\nCV Results: acc={mean_acc:.3f}  f1={mean_f1:.3f}")
        print(classification_report(all_y_true, all_y_pred,
              target_names=self.le_target.classes_, zero_division=0))

        # Train final model on all data
        print("Training final model on all data...")
        self.model = XGBClassifier(
            n_estimators=300, max_depth=5, learning_rate=0.05,
            subsample=0.80, colsample_bytree=0.70, min_child_weight=5,
            gamma=1.0, reg_alpha=0.10, reg_lambda=1.0,
            objective='multi:softprob', num_class=len(self.le_target.classes_),
            eval_metric='mlogloss', random_state=42, n_jobs=-1,
        )
        self.model.fit(X.values, y, sample_weight=sample_weights, verbose=False)
        self.trained = True

        # Save
        with open(save_path, 'wb') as f:
            pickle.dump({'model': self.model, 'le': self.le_target,
                         'feature_cols': self.feature_cols}, f)
        print(f"Saved: {save_path}")
        return mean_acc, mean_f1

    def load(self, path: str = 'exercise_classifier.pkl'):
        with open(path, 'rb') as f:
            d = pickle.load(f)
        self.model        = d['model']
        self.le_target    = d['le']
        self.feature_cols = d['feature_cols']
        self.trained      = True
        print(f"Exercise classifier loaded from {path}")
        print(f"  Classes: {list(self.le_target.classes_)}")

    def predict(self, scalar_dict: dict) -> str:
        """
        Predict exercise from a dict of scalar features (one rep).
        Returns exercise name string.
        scalar_dict keys must match the base feature columns from final_cleaned_data.csv
        """
        if not self.trained:
            print("  WARNING: classifier not trained — returning 'unknown'")
            return 'unknown'
        try:
            row = pd.DataFrame([scalar_dict])
            # Engineer features
            row_eng = engineer_features(row)
            # Align to training feature columns
            for col in self.feature_cols:
                if col not in row_eng.columns:
                    row_eng[col] = 0.0
            row_eng = row_eng[self.feature_cols].fillna(0.0)
            pred_idx = int(self.model.predict(row_eng.values)[0])
            return str(self.le_target.inverse_transform([pred_idx])[0])
        except Exception as e:
            print(f"  Exercise prediction failed: {e}")
            return 'unknown'

    def predict_from_scalars_array(self, scalars_10: np.ndarray) -> str:
        """
        Predict exercise from the 10-dim scalar array produced by extract_scalars().
        Maps to approximate feature names for compatibility.
        """
        # Our 10-dim scalars: [l_rom, r_rom, l_peak, r_peak, sym, vel, jerk, trunk, lag, smooth]
        # Map to rough feature names for the classifier
        scalar_dict = {
            'left_knee_ROM':          float(scalars_10[0]) * 180,
            'right_knee_ROM':         float(scalars_10[1]) * 180,
            'left_knee_peak_angle':   float(scalars_10[2]) * 180,
            'right_knee_peak_angle':  float(scalars_10[3]) * 180,
            'left_knee_mean_velocity': float(scalars_10[5]) * 200,
            'right_knee_mean_velocity': float(scalars_10[5]) * 200,
            'left_hip_ROM':           float(scalars_10[0]) * 150,
            'right_hip_ROM':          float(scalars_10[1]) * 150,
            'left_ankle_ROM':         float(scalars_10[0]) * 60,
            'right_ankle_ROM':        float(scalars_10[1]) * 60,
        }
        return self.predict(scalar_dict)


# Global instance — loaded once, reused across reps
_exercise_clf = ExerciseClassifier()

def load_exercise_classifier(path='exercise_classifier.pkl'):
    global _exercise_clf
    if os.path.exists(path):
        _exercise_clf.load(path)
    else:
        print(f"No saved classifier at {path} — will use neural head fallback")

print("Exercise classifier module ready.")
print("To train: _exercise_clf.train(csv_path='final_cleaned_data.csv')")
print("To load:  load_exercise_classifier('exercise_classifier.pkl')")


Exercise classifier module ready.
To train: _exercise_clf.train(csv_path='final_cleaned_data.csv')
To load:  load_exercise_classifier('exercise_classifier.pkl')


In [5]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  FULL PIPELINE — Paste entire file into ONE Colab cell                     ║
# ║                                                                             ║
# ║  FLOW:                                                                      ║
# ║  Video → PS1 (pipeline10 steps) → PS2 (RehabNet) → quality + torque        ║
# ║                                                                             ║
# ║  SETUP: run this one cell first:                                            ║
# ║    !pip install mediapipe==0.10.21 opencv-contrib-python scipy -q           ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import cv2, numpy as np, pandas as pd, os, re, time, copy, urllib.request
import torch, torch.nn as nn, torch.nn.functional as F
from scipy.signal import savgol_filter
from scipy.interpolate import interp1d
from typing import List, Optional
from collections import Counter, deque

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 1 — SHARED CONSTANTS
# ══════════════════════════════════════════════════════════════════════════════

TARGET_LEN        = 150
N_JOINTS          = 6
IN_CHANNELS       = 3
MIN_FRAMES        = 60
FPS_DEFAULT       = 30
N_EPOCHS          = 60

# pipeline10 constants (exact values from their notebook)
MODEL_COMPLEXITY  = 1
MIN_DETECT_CONF   = 0.3
MIN_TRACK_CONF    = 0.3
VISIBILITY_THRESH = 0.4
UPSCALE_W, UPSCALE_H = 1440, 1080
CROP_X = (0.20, 0.80)
CROP_Y = (0.05, 0.95)
CLAHE_CLIP        = 4.0
CLAHE_GRID        = (8, 8)
SAVGOL_WINDOW     = 7
SAVGOL_ORDER      = 3
MAX_INTERP_GAP    = 5
VEL_SPIKE_THRESH  = 300
ANAT_LIMITS = {
    'angle_knee_L': (0,175),  'angle_knee_R': (0,175),
    'angle_hip_L':  (-30,140),'angle_hip_R':  (-30,140),
    'angle_ankle_L':(50,140), 'angle_ankle_R':(50,140),
}

# pipeline10 joint definitions (exact from their notebook)
CANONICAL_JOINTS = {
    'HipLeft':23,'HipRight':24,'KneeLeft':25,'KneeRight':26,
    'AnkleLeft':27,'AnkleRight':28,'FootLeft':31,'FootRight':32,
    'ShoulderLeft':11,'ShoulderRight':12,
}
MIDPOINT_JOINTS = {
    'SpineBase':('HipLeft','HipRight'),
    'SpineMid': ('ShoulderLeft','ShoulderRight'),
}
ALL_JOINTS = list(CANONICAL_JOINTS) + list(MIDPOINT_JOINTS)
JOINT_COLS = [f'{j}_{ax}' for j in ALL_JOINTS for ax in ('x','y','z')]
BASE_COLS  = ['subject_id','session','label','frame_id','time_s','detection']

# joint order for RehabNet (6 joints we care about)
JOINT_ORDER = ['l_hip','l_knee','l_ankle','r_hip','r_knee','r_ankle']
JOINT_MAP   = {   # maps JOINT_ORDER name → pipeline10 column prefix
    'l_hip':'HipLeft','l_knee':'KneeLeft','l_ankle':'AnkleLeft',
    'r_hip':'HipRight','r_knee':'KneeRight','r_ankle':'AnkleRight',
}

EXERCISE_LIST = [
    'deep_squat','hurdle_step','inline_lunge','side_lunge',
    'sit_to_stand','straight_leg_raise','squat','ctk_squat','unknown'
]
EXERCISE2IDX = {e:i for i,e in enumerate(EXERCISE_LIST)}
N_EXERCISES  = len(EXERCISE_LIST)

HARDWARE_MODES = {
    0:{'name':'free',              'scale':0.0, 'bias':0.0, 'min':0.0,'max':0.0},
    1:{'name':'constant_torque',   'scale':0.0, 'bias':0.5, 'min':0.4,'max':0.6},
    2:{'name':'resistive_torque',  'scale':0.5, 'bias':0.0, 'min':0.0,'max':0.8},
    3:{'name':'increasing_torque', 'scale':1.0, 'bias':0.0, 'min':0.0,'max':1.0},
    4:{'name':'dampening',         'scale':-0.5,'bias':0.5, 'min':0.0,'max':0.5},
}
N_MODES = len(HARDWARE_MODES)

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 2 — PS1: VIDEO → JOINT DATAFRAME
# (exact functions from pipeline10 notebook)
# ══════════════════════════════════════════════════════════════════════════════

_clahe_obj = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_GRID)

def ps1_enhance(frame):
    l, a, b = cv2.split(cv2.cvtColor(frame, cv2.COLOR_BGR2LAB))
    return cv2.cvtColor(cv2.merge([_clahe_obj.apply(l), a, b]), cv2.COLOR_LAB2BGR)

def ps1_prepare_frame(frame):
    big = cv2.resize(frame, (UPSCALE_W, UPSCALE_H), interpolation=cv2.INTER_LINEAR)
    h, w = big.shape[:2]
    x1,y1 = int(w*CROP_X[0]), int(h*CROP_Y[0])
    x2,y2 = int(w*CROP_X[1]), int(h*CROP_Y[1])
    return ps1_enhance(big[y1:y2, x1:x2])

def ps1_get_xyz(lm, idx):
    """Works with both solutions API landmark list and Tasks API landmark list."""
    p = lm[idx]
    vis = getattr(p, 'visibility', 1.0)
    return (p.x, p.y, p.z) if vis >= VISIBILITY_THRESH else None

def ps1_extract_row(lm):
    coords = {j: ps1_get_xyz(lm, idx) for j, idx in CANONICAL_JOINTS.items()}
    for j, (j1, j2) in MIDPOINT_JOINTS.items():
        a, b = coords.get(j1), coords.get(j2)
        coords[j] = tuple((a[i]+b[i])/2 for i in range(3)) if a and b else None
    row = {}
    for j in ALL_JOINTS:
        xyz = coords[j]
        for i, ax in enumerate(('x','y','z')):
            row[f'{j}_{ax}'] = xyz[i] if xyz else float('nan')
    return row

def ps1_mask_spikes(s, fps):
    s = s.copy().astype(float)
    s[s.diff().abs() * fps > VEL_SPIKE_THRESH] = float('nan')
    return s

def ps1_interp_gap(s, max_gap=MAX_INTERP_GAP):
    s = s.copy().astype(float)
    mask = s.isna()
    if not mask.any(): return s
    run_id  = mask.ne(mask.shift()).cumsum()
    run_len = mask.groupby(run_id).transform('sum')
    si = s.interpolate(method='linear', limit_direction='both')
    si[mask & (run_len > max_gap)] = float('nan')
    return si

def ps1_root_center(df):
    df = df.copy()
    for j in ALL_JOINTS:
        for ax in ('x','y','z'):
            df[f'{j}_{ax}'] -= df[f'SpineBase_{ax}']
    return df

def ps1_bone_normalize(df):
    df = df.copy()
    det = df['detection'] == 1
    heights = []
    for hip, knee, ankle in [('HipLeft','KneeLeft','AnkleLeft'),
                              ('HipRight','KneeRight','AnkleRight')]:
        segs = []
        for j1, j2 in [(hip,knee),(knee,ankle),('SpineBase','SpineMid')]:
            seg = np.sqrt(sum((df[f'{j1}_{ax}']-df[f'{j2}_{ax}'])**2 for ax in 'xyz'))
            segs.append(seg)
        h = (segs[0]+segs[1]+segs[2])[det].median()
        if not np.isnan(h): heights.append(h)
    body_h = np.mean(heights) if heights else 1.0
    df[JOINT_COLS] = df[JOINT_COLS] / max(body_h, 1e-6)
    return df

def ps1_angle_3d(df, a, b, c):
    A = df[[f'{a}_x',f'{a}_y',f'{a}_z']].values
    B = df[[f'{b}_x',f'{b}_y',f'{b}_z']].values
    C = df[[f'{c}_x',f'{c}_y',f'{c}_z']].values
    ba, bc = A-B, C-B
    n   = np.linalg.norm(ba,axis=1)*np.linalg.norm(bc,axis=1)
    dot = np.einsum('ij,ij->i', ba, bc)
    with np.errstate(invalid='ignore', divide='ignore'):
        ang = np.degrees(np.arccos(np.clip(np.where(n>0,dot/n,0),-1,1)))
    ang[n==0] = np.nan
    return pd.Series(ang, index=df.index)

def ps1_extract_features(df):
    df  = df.copy()
    dt  = df['time_s'].diff().median()
    fps = 1.0/dt if dt and dt > 0 else FPS_DEFAULT
    _sw = max(SAVGOL_ORDER+2, round(SAVGOL_WINDOW/30.0*fps))
    sw  = _sw if _sw%2==1 else _sw+1
    ANGLE_DEFS = [
        ('angle_knee_L','HipLeft','KneeLeft','AnkleLeft'),
        ('angle_knee_R','HipRight','KneeRight','AnkleRight'),
        ('angle_hip_L','SpineBase','HipLeft','KneeLeft'),
        ('angle_hip_R','SpineBase','HipRight','KneeRight'),
        ('angle_ankle_L','KneeLeft','AnkleLeft','FootLeft'),
        ('angle_ankle_R','KneeRight','AnkleRight','FootRight'),
    ]
    ang_cols = [d[0] for d in ANGLE_DEFS]
    for name, a, b, c in ANGLE_DEFS:
        lo, hi = ANAT_LIMITS.get(name,(0,360))
        df[name] = ps1_angle_3d(df,a,b,c).clip(lo,hi)
    for col in ang_cols:
        df[col] = ps1_mask_spikes(df[col], fps)
        df[col] = ps1_interp_gap(df[col])
    for col in ang_cols:
        s, sc = df[col].dropna(), col+'_smooth'
        if len(s) >= sw:
            sm = pd.Series(np.nan, index=df.index)
            sm[s.index] = savgol_filter(s.values, sw, SAVGOL_ORDER)
            df[sc] = sm
        else:
            df[sc] = df[col]
    return df

def ps1_clean_nulls(df, excl=('subject_id','session','label')):
    df = df.copy()
    for col in [c for c in df.columns if c not in excl]:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        if df[col].isna().any():
            df[col] = (df[col].interpolate(method='linear',
                       limit_direction='both').ffill().bfill().fillna(0))
    return df

def ps1_process_video(video_path, patient_id, exercise='unknown',
                      session='1', label='unknown'):
    """
    VIDEO → processed DataFrame (same as pipeline10 Cell 15+17).

    Returns DataFrame with columns:
      frame_id, time_s, detection,
      HipLeft_x/y/z, KneeLeft_x/y/z, AnkleLeft_x/y/z,
      HipRight_x/y/z, KneeRight_x/y/z, AnkleRight_x/y/z,
      angle_knee_L_smooth, angle_knee_R_smooth, ...
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open: {video_path}")

    fps_vid   = cap.get(cv2.CAP_PROP_FPS) or FPS_DEFAULT
    rows      = []
    frame_idx = 0
    detected  = 0
    t0        = time.time()

    # ── MediaPipe Tasks API (works with protobuf 5.x / tensorflow 2.20) ──────
    import mediapipe as mp
    from mediapipe.tasks import python as _mptasks
    from mediapipe.tasks.python import vision as _mpvision

    _MODEL_PATH = '/tmp/pose_landmarker.task'
    if not os.path.exists(_MODEL_PATH):
        print("  Downloading pose landmarker model (~30MB)...")
        urllib.request.urlretrieve(
            'https://storage.googleapis.com/mediapipe-models/pose_landmarker/'
            'pose_landmarker_heavy/float16/latest/pose_landmarker_heavy.task',
            _MODEL_PATH
        )
        print("  Downloaded.")

    _base_opts = _mptasks.BaseOptions(model_asset_path=_MODEL_PATH)
    _pose_opts = _mpvision.PoseLandmarkerOptions(
        base_options              = _base_opts,
        running_mode              = _mpvision.RunningMode.VIDEO,
        num_poses                 = 1,
        min_pose_detection_confidence = MIN_DETECT_CONF,
        min_pose_presence_confidence  = MIN_DETECT_CONF,
        min_tracking_confidence       = MIN_TRACK_CONF,
        output_segmentation_masks = False,
    )

    with _mpvision.PoseLandmarker.create_from_options(_pose_opts) as pose:
        frame_ms = 0
        frame_step_ms = int(1000.0 / fps_vid)
        while True:
            ok, frame = cap.read()
            if not ok: break
            frame_idx += 1
            frame_ms = frame_idx * frame_step_ms
            prepped  = ps1_prepare_frame(frame)
            rgb      = cv2.cvtColor(prepped, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            res      = pose.detect_for_video(mp_image, frame_ms)
            row = {
                'subject_id': patient_id, 'session': session,
                'label': label, 'frame_id': frame_idx,
                'time_s': round(frame_idx/fps_vid, 4), 'detection': 0,
            }
            row.update({col: float('nan') for col in JOINT_COLS})
            if res.pose_landmarks and len(res.pose_landmarks) > 0:
                detected += 1
                row['detection'] = 1
                row.update(ps1_extract_row(res.pose_landmarks[0]))
            rows.append(row)
    cap.release()

    if frame_idx == 0:
        raise ValueError("Empty video")

    det_pct = 100 * detected / frame_idx
    print(f"  PS1: {frame_idx} frames  detection={det_pct:.0f}%  {time.time()-t0:.1f}s")
    if det_pct < 20:
        print(f"  WARNING: low detection ({det_pct:.0f}%) — check video quality/angle")

    df = pd.DataFrame(rows, columns=BASE_COLS+JOINT_COLS)
    df = df[df['detection']==1].reset_index(drop=True)
    df = ps1_root_center(df)
    df = ps1_bone_normalize(df)
    df = ps1_extract_features(df)
    df = ps1_clean_nulls(df)
    return df

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 3 — PS1→PS2 BRIDGE: DataFrame → RehabNet samples
# ══════════════════════════════════════════════════════════════════════════════

def _build_adj():
    A = np.zeros((N_JOINTS, N_JOINTS), dtype=np.float32)
    for i, j in [(0,1),(1,2),(3,4),(4,5),(0,3),(1,4),(2,5)]:
        A[i,j] = A[j,i] = 1.0
    np.fill_diagonal(A, 1.0)
    d = np.diag(1.0/np.sqrt(A.sum(axis=1)+1e-6))
    return (d @ A @ d).astype(np.float32)

ADJ = _build_adj()

def _df_to_kps(df):
    """Extract (T, 6, 3) xyz from processed DataFrame."""
    arr = np.zeros((len(df), N_JOINTS, 3), dtype=np.float32)
    for ji, jname in enumerate(JOINT_ORDER):
        mp_name = JOINT_MAP[jname]
        for ci, ax in enumerate(('x','y','z')):
            col = f'{mp_name}_{ax}'
            arr[:, ji, ci] = df[col].values if col in df.columns else 0.0
    return np.nan_to_num(arr, nan=0.0)

def _process_kps(kps):
    """(T,6,3) → (TARGET_LEN,6,3) smoothed + normalised."""
    T = len(kps)
    if T < MIN_FRAMES:
        raise ValueError(f"Too short: {T} frames")
    win = min(11, T if T%2==1 else T-1)
    win = max(win, 5)
    out_raw = kps.copy()
    if T >= win:
        for j in range(N_JOINTS):
            for c in range(3):
                out_raw[:,j,c] = savgol_filter(kps[:,j,c], win, 3)
    out = np.zeros((TARGET_LEN, N_JOINTS, 3), dtype=np.float32)
    for j in range(N_JOINTS):
        for c in range(3):
            out[:,j,c] = np.interp(np.linspace(0,1,TARGET_LEN),
                                   np.linspace(0,1,T), out_raw[:,j,c])
    hip   = ((out[:,0,:]+out[:,3,:])/2.0).mean(axis=0)
    out   = out - hip
    l     = np.mean(np.linalg.norm(out[:,0,:]-out[:,2,:], axis=1))
    r     = np.mean(np.linalg.norm(out[:,3,:]-out[:,5,:], axis=1))
    scale = (l+r)/2.0
    if scale < 1e-4: scale = 1.0
    out   = np.clip(out/scale, -5.0, 5.0)
    return np.nan_to_num(out, nan=0.0).astype(np.float32)

def _extract_scalars(kps, df=None):
    """Build 10-scalar vector. Uses pipeline10 smoothed angles if df provided."""
    def _ang(a, b, c):
        ba=a-b; bc=c-b
        cos=np.sum(ba*bc,1)/(np.linalg.norm(ba,1)*np.linalg.norm(bc,1)+1e-6)
        return np.degrees(np.arccos(np.clip(cos,-1,1)))

    if df is not None and 'angle_knee_L_smooth' in df.columns:
        lk = df['angle_knee_L_smooth'].ffill().bfill().values.astype(float)
        rk = df['angle_knee_R_smooth'].ffill().bfill().values.astype(float)
    else:
        lk = _ang(kps[:,0,:], kps[:,1,:], kps[:,2,:])
        rk = _ang(kps[:,3,:], kps[:,4,:], kps[:,5,:])

    lr  = float(np.nanmax(lk)-np.nanmin(lk))
    rr  = float(np.nanmax(rk)-np.nanmin(rk))
    sym = abs(lr-rr)/((lr+rr)/2+1e-6)*100 if (lr+rr)>1 else 0.0

    fps = FPS_DEFAULT
    if df is not None and 'time_s' in df.columns:
        dt = df['time_s'].diff().median()
        if dt and dt > 0: fps = float(np.clip(1.0/dt, 1, 300))

    vel  = float(np.nanmax(np.abs(np.diff(lk))*fps)) if len(lk)>1 else 0.0
    hip  = (kps[:,0,:]+kps[:,3,:])/2.0
    jerk = float(np.nanmean(np.linalg.norm(np.diff(np.diff(hip,axis=0),axis=0),axis=1)))
    peak = int(np.argmin(lk))
    trunk= float(abs(hip[peak,0]-hip[0,0]))
    lk_c = lk-np.nanmean(lk); rk_c = rk-np.nanmean(rk)
    lag  = float(abs(np.argmax(np.correlate(lk_c,rk_c,'full'))-(len(lk)-1))) \
           if np.nanstd(lk_c)>1e-6 and np.nanstd(rk_c)>1e-6 else 0.0
    smooth = float(np.clip(1.0/(np.nanstd(np.diff(lk))+1e-6)/100, 0, 1))

    sc = np.array([
        np.clip(lr/180,0,1),  np.clip(rr/180,0,1),
        np.clip(np.nanmin(lk)/180,0,1), np.clip(np.nanmin(rk)/180,0,1),
        np.clip(sym/100,0,5), np.clip(vel/200,0,5),
        np.clip(jerk/10,0,5), np.clip(trunk,0,5),
        np.clip(lag/TARGET_LEN,0,1), smooth,
    ], dtype=np.float32)
    return np.nan_to_num(sc, nan=0.0, posinf=1.0, neginf=0.0)

def _segment_reps(df, angle_col='angle_knee_L_smooth',
                  min_frames=MIN_FRAMES, min_rom=20.0):
    """
    Split continuous DataFrame into per-rep DataFrames.
    Adaptive: automatically adjusts spacing based on video length
    so it works correctly regardless of how many reps are in the video.
    """
    if angle_col in df.columns:
        angles = df[angle_col].ffill().bfill().values
    else:
        kps = _df_to_kps(df)
        hip=kps[:,0,:]; knee=kps[:,1,:]; ankle=kps[:,2,:]
        v1=hip-knee; v2=ankle-knee
        cos=np.sum(v1*v2,1)/(np.linalg.norm(v1,1)*np.linalg.norm(v2,1)+1e-6)
        angles = np.degrees(np.arccos(np.clip(cos,-1,1)))

    n = len(angles)

    # Adaptive min_frames: at least 60 frames but scale with video length
    # For a 20s video at 30fps = 600 frames, expect ~5 reps → ~120 frames/rep
    # For a 60s video at 30fps = 1800 frames, expect ~15 reps → ~120 frames/rep
    adaptive_min = max(min_frames, n // 20)  # never allow more than 20 reps
    half = adaptive_min // 2

    # Find valleys (peak knee bend = deepest angle = minimum value)
    from scipy.signal import find_peaks
    # Invert angles so valleys become peaks for find_peaks
    inv_angles = -angles
    peaks, props = find_peaks(
        inv_angles,
        distance=adaptive_min,          # minimum spacing between reps
        prominence=min_rom * 0.5,       # must be a real movement, not noise
        width=adaptive_min // 3,        # rep must have some duration
    )

    if len(peaks) == 0:
        # Fallback: return whole video as one rep
        return [df]

    # Build rep boundaries: midpoints between consecutive valleys
    midpoints = []
    for i in range(len(peaks) - 1):
        mid = (peaks[i] + peaks[i+1]) // 2
        midpoints.append(mid)

    bounds = [0] + midpoints + [n]
    reps = []
    for i in range(len(bounds) - 1):
        rep_df = df.iloc[bounds[i]:bounds[i+1]].copy()
        if len(rep_df) >= adaptive_min:
            reps.append(rep_df)

    return reps if reps else [df]

def ps1_df_to_samples(df, patient_id, exercise='unknown',
                      label=-1, source='patient_video'):
    """
    PS1 DataFrame → list of sample dicts ready for RehabNet.
    This is the PS1→PS2 bridge.
    """
    reps    = _segment_reps(df)
    samples = []
    ex_idx  = EXERCISE2IDX.get(exercise, EXERCISE2IDX['unknown'])

    for rep_i, rep_df in enumerate(reps):
        try:
            kps_raw = _df_to_kps(rep_df)
            kps     = _process_kps(kps_raw)
            scalars = _extract_scalars(kps, rep_df)
            samples.append({
                'keypoints':    kps,
                'adj':          ADJ,
                'label':        int(max(label, 0)),
                'quality':      float(max(label, 0)),
                'exercise':     exercise,
                'exercise_idx': ex_idx,
                'subject':      f'patient_{patient_id}',
                'source':       source,
                'scalars':      scalars,
                'rep_id':       f'{patient_id}_r{rep_i+1}',
            })
        except Exception as e:
            print(f"  [Bridge] rep {rep_i+1} failed: {e}")

    print(f"  Bridge: {len(reps)} reps found → {len(samples)} valid samples")
    return samples

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 4 — PS2: REHABNET (Transformer-LSTM)
# ══════════════════════════════════════════════════════════════════════════════

def _smooth(arr, win=7, poly=2):
    arr = np.nan_to_num(arr)
    if len(arr) < win+2: return arr
    w = win if win%2==1 else win+1
    w = max(w, poly+2 if (poly+2)%2==1 else poly+3)
    try: return savgol_filter(arr, w, poly)
    except: return arr

def _resample(arr, n):
    arr = np.nan_to_num(arr)
    if len(arr)==n: return arr
    if len(arr)<2: return np.zeros(n, dtype=np.float32)
    return np.interp(np.linspace(0,1,n), np.linspace(0,1,len(arr)), arr)

class STGCNBlock(nn.Module):
    def __init__(self, c_in, c_out, ks=9, stride=1):
        super().__init__()
        self.gcn = nn.Linear(c_in, c_out)
        self.tcn = nn.Sequential(
            nn.BatchNorm2d(c_out), nn.ReLU(),
            nn.Conv2d(c_out,c_out,(ks,1),(stride,1),((ks-1)//2,0)),
            nn.BatchNorm2d(c_out), nn.Dropout(0.1))
        self.res = (nn.Sequential(nn.Conv2d(c_in,c_out,1,(stride,1)),
                                   nn.BatchNorm2d(c_out))
                    if c_in!=c_out or stride!=1 else nn.Identity())
        self.relu = nn.ReLU()

    def forward(self, x, A):
        B,C,T,V = x.shape
        if A.dim()==3: A=A[0]
        xs = x.permute(0,2,3,1).contiguous().view(B*T,V,C)
        xs = torch.bmm(A.unsqueeze(0).expand(B*T,-1,-1), xs)
        xs = self.gcn(xs).view(B,T,V,-1).permute(0,3,1,2)
        t_out = self.tcn(xs); r_out = self.res(x)
        if torch.isnan(t_out).any() or torch.isnan(r_out).any():
            t_out = torch.nan_to_num(t_out); r_out = torch.nan_to_num(r_out)
        return self.relu(t_out + r_out)

class STGCN(nn.Module):
    def __init__(self):
        super().__init__()
        self.input_norm = nn.LayerNorm(IN_CHANNELS*N_JOINTS)
        self.b1 = STGCNBlock(3,32); self.b2 = STGCNBlock(32,64); self.b3 = STGCNBlock(64,128)

    def forward(self, x, A):
        B,C,T,V = x.shape
        x  = torch.clamp(x,-5.,5.)
        xf = x.permute(0,2,1,3).contiguous().view(B*T,C*V)
        xf = self.input_norm(xf).view(B,T,C,V).permute(0,2,1,3)
        return self.b3(self.b2(self.b1(xf,A),A),A).mean(3).permute(0,2,1)

class ClinicalTransformer(nn.Module):
    def __init__(self, d=128, heads=4, d_ff=256, drop=0.1):
        super().__init__()
        pe  = torch.zeros(TARGET_LEN+1,d)
        pos = torch.arange(TARGET_LEN+1).unsqueeze(1).float()
        div = torch.exp(torch.arange(0,d,2).float()*(-np.log(10000.)/d))
        pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div)
        self.register_buffer('pe', pe.unsqueeze(0))
        self.cls  = nn.Parameter(torch.randn(1,1,d)*0.02)
        self.drop = nn.Dropout(drop)
        mk = lambda: nn.TransformerEncoderLayer(d,heads,d_ff,drop,batch_first=True,norm_first=True)
        self.l1,self.l2 = mk(),mk()
        self.n1,self.n2 = nn.LayerNorm(d),nn.LayerNorm(d)

    def forward(self, x, return_attn=False):
        B,T,_ = x.shape
        x = torch.cat([self.cls.expand(B,-1,-1),x],1)
        x = self.drop(x+self.pe[:,:T+1,:])
        x = self.n1(self.l1(x))
        if return_attn:
            sa=self.l2.self_attn; xn=self.l2.norm1(x)
            out,attn = sa(xn,xn,xn,need_weights=True,average_attn_weights=True)
            x2=x+self.l2.dropout1(out); xn2=self.l2.norm2(x2)
            ff=self.l2.linear2(self.l2.dropout(self.l2.activation(self.l2.linear1(xn2))))
            return self.n2(x2+self.l2.dropout2(ff))[:,0,:], attn
        return self.n2(self.l2(x))[:,0,:], None

class RehabNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.stgcn        = STGCN()
        self.trans        = ClinicalTransformer()
        self.mode_emb     = nn.Embedding(N_MODES,16)
        self.exercise_emb = nn.Embedding(N_EXERCISES,8)
        self.shared       = nn.Sequential(
            nn.Linear(128+10+16+8,128), nn.LayerNorm(128), nn.ReLU(), nn.Dropout(0.2))
        self.classify_head = nn.Linear(128,2)
        self.quality_head  = nn.Linear(128,1)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def encode(self, kps, adj, return_attn=False):
        if adj.dim()==3: adj=adj[0]
        return self.trans(self.stgcn(kps,adj), return_attn)

    def forward(self, kps, adj, scalars, mode=None, exercise_idx=None):
        B = kps.shape[0]
        if mode is None: mode=torch.zeros(B,dtype=torch.long,device=kps.device)
        if exercise_idx is None: exercise_idx=torch.zeros(B,dtype=torch.long,device=kps.device)
        cls,_ = self.encode(kps,adj)
        x = self.shared(torch.cat([cls,scalars,self.mode_emb(mode),self.exercise_emb(exercise_idx)],1))
        return self.classify_head(x), self.quality_head(x).squeeze(1)

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 5 — TRAINING (LOSO)
# ══════════════════════════════════════════════════════════════════════════════

from torch.utils.data import Dataset, DataLoader

class RehabDataset(Dataset):
    def __init__(self, samples, augment=False):
        self.samples=samples; self.augment=augment
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        s   = self.samples[idx]
        kps = np.nan_to_num(s['keypoints'].copy(), nan=0.0)
        kps = np.clip(kps,-5.,5.)
        if self.augment: kps = self._aug(kps)
        return {
            'keypoints':    torch.from_numpy(kps).permute(2,0,1).float(),
            'adj':          torch.from_numpy(s['adj']).float(),
            'label':        torch.tensor(s['label'],dtype=torch.long),
            'quality':      torch.tensor(s['quality'],dtype=torch.float32),
            'scalars':      torch.from_numpy(np.nan_to_num(s['scalars'])).float(),
            'exercise_idx': torch.tensor(s['exercise_idx'],dtype=torch.long),
            'exercise':     s['exercise'], 'subject': s['subject'],
        }
    def _aug(self, kps):
        if np.random.rand()<0.5:
            n=max(int(TARGET_LEN*np.random.uniform(0.8,1.2)),MIN_FRAMES)
            tmp=np.zeros((n,N_JOINTS,3),dtype=np.float32)
            for j in range(N_JOINTS):
                for c in range(3): tmp[:,j,c]=_resample(kps[:,j,c],n)
            kps=np.zeros((TARGET_LEN,N_JOINTS,3),dtype=np.float32)
            for j in range(N_JOINTS):
                for c in range(3): kps[:,j,c]=_resample(tmp[:,j,c],TARGET_LEN)
        if np.random.rand()<0.5:
            kps[:,[0,1,2,3,4,5],:]=kps[:,[3,4,5,0,1,2],:]
            kps[:,:,0]*=-1.
        if np.random.rand()<0.5:
            kps+=np.random.randn(*kps.shape).astype(np.float32)*0.02
        if np.random.rand()<0.3:
            th=np.radians(np.random.uniform(-10,10))
            x=kps[:,:,0].copy(); z=kps[:,:,2].copy()
            kps[:,:,0]=np.cos(th)*x-np.sin(th)*z
            kps[:,:,2]=np.sin(th)*x+np.cos(th)*z
        return np.clip(kps,-5.,5.)

def _collate(batch):
    return {k: torch.stack([b[k] for b in batch])
              if isinstance(batch[0][k],torch.Tensor)
              else [b[k] for b in batch] for k in batch[0]}

def _build_loaders(train, test, batch_size=32):
    from torch.utils.data import WeightedRandomSampler
    ex_counts = Counter(s['exercise'] for s in train)
    total     = len(train)
    weights   = [total/ex_counts[s['exercise']] for s in train]
    sampler   = WeightedRandomSampler(weights, len(train), replacement=True)
    return (DataLoader(RehabDataset(train,True), batch_size,
                       sampler=sampler, collate_fn=_collate, num_workers=2),
            DataLoader(RehabDataset(test,False), batch_size, False,
                       collate_fn=_collate, num_workers=2))

def _class_weights(train_samples, device):
    n0 = sum(s['label']==0 for s in train_samples)
    n1 = sum(s['label']==1 for s in train_samples)
    tot = n0+n1
    return torch.tensor([tot/(2*n0+1e-6), tot/(2*n1+1e-6)],
                        dtype=torch.float32, device=device)

def _train_epoch(model, loader, opt, device, cw, epoch=0):
    model.train()
    A=torch.from_numpy(ADJ).to(device)
    total=0.; used=0; skipped=0
    for bi, b in enumerate(loader):
        kps=b['keypoints'].to(device); lbl=b['label'].to(device)
        qual=b['quality'].to(device);  sc=b['scalars'].to(device)
        ex_idx=b['exercise_idx'].to(device)
        good=(torch.isfinite(kps).flatten(1).all(1) &
              torch.isfinite(sc).all(1) & torch.isfinite(qual))
        if not good.all():
            kps=kps[good]; lbl=lbl[good]; qual=qual[good]
            sc=sc[good]; ex_idx=ex_idx[good]
        if kps.shape[0]==0: skipped+=1; continue
        mode=torch.zeros(kps.shape[0],dtype=torch.long,device=device)
        logits,qpred=model(kps,A,sc,mode,ex_idx)
        if not torch.isfinite(logits).all() or not torch.isfinite(qpred).all():
            skipped+=1; continue
        loss=(F.cross_entropy(logits,lbl,weight=cw) +
              0.3*F.mse_loss(torch.sigmoid(qpred),torch.clamp(qual,0,1)))
        if not torch.isfinite(loss): skipped+=1; continue
        opt.zero_grad(set_to_none=True); loss.backward()
        if epoch==0 and bi==0:
            gn=sum(p.grad.norm()**2 for p in model.parameters()
                   if p.grad is not None)**0.5
            print(f"  [ep0] grad norm={gn:.3f}")
        nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        opt.step(); total+=loss.item(); used+=1
    if skipped>0: print(f"  epoch {epoch}: skipped {skipped} batches")
    return total/max(used,1)

@torch.no_grad()
def _evaluate(model, loader, device):
    model.eval()
    A=torch.from_numpy(ADJ).to(device)
    preds,labels,quals,qpreds=[],[],[],[]
    for b in loader:
        kps=b['keypoints'].to(device); sc=b['scalars'].to(device)
        ex_idx=b['exercise_idx'].to(device); qual=b['quality'].to(device)
        lbl=b['label'].to(device)
        good=(torch.isfinite(kps).flatten(1).all(1) &
              torch.isfinite(sc).all(1) & torch.isfinite(qual))
        if good.sum()==0: continue
        kps=kps[good]; sc=sc[good]; ex_idx=ex_idx[good]
        qual=qual[good]; lbl=lbl[good]
        mode=torch.zeros(kps.shape[0],dtype=torch.long,device=device)
        lg,qp=model(kps,A,sc,mode,ex_idx)
        preds.extend(lg.argmax(1).cpu().numpy())
        labels.extend(lbl.cpu().numpy())
        quals.extend(qual.cpu().numpy())
        qpreds.extend(torch.sigmoid(qp).cpu().numpy())
    from sklearn.metrics import accuracy_score,f1_score,classification_report
    from scipy.stats import pearsonr
    if not labels: return {'acc':0.,'f1':0.,'quality_r':0.}
    acc=accuracy_score(labels,preds)
    f1=f1_score(labels,preds,average='macro',zero_division=0)
    r=pearsonr(quals,qpreds)[0] if len(set(quals))>1 else 0.
    print(classification_report(labels,preds,
          target_names=['incorrect','correct'],zero_division=0))
    print(f"  quality r={r:.3f}")
    return {'acc':acc,'f1':f1,'quality_r':r}

def run_loso(samples, n_epochs=N_EPOCHS, batch_size=32, lr=3e-4, patience=12):
    """
    Train RehabNet with Leave-One-Subject-Out cross validation.
    samples : output of build_master_dataset() or ps1_df_to_samples()
    """
    device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")
    subjects = sorted({s['subject'] for s in samples if s['source']!='keraal'
                       and 'uiprmd' in s['subject']})
    if not subjects:
        subjects = sorted({s['subject'] for s in samples})
    results  = []

    for subj in subjects:
        print(f"\n{'='*50}\nLOSO held-out: {subj}\n{'='*50}")
        tr = [s for s in samples if s['subject']!=subj]
        te = [s for s in samples if s['subject']==subj]
        tl,vl = _build_loaders(tr,te,batch_size)
        cw    = _class_weights(tr,device)
        print(f"  weights: incorrect={cw[0]:.3f} correct={cw[1]:.3f}")

        model = RehabNet().to(device)
        opt   = torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-4)
        def lr_fn(ep):
            w=5
            if ep<w: return (ep+1)/w
            return 0.5*(1+np.cos(np.pi*(ep-w)/max(n_epochs-w,1)))
        sch = torch.optim.lr_scheduler.LambdaLR(opt,lr_fn)

        best_f1=0.; best_state=None; no_imp=0
        for ep in range(n_epochs):
            loss=_train_epoch(model,tl,opt,device,cw,epoch=ep)
            sch.step()
            if (ep+1)%5==0:
                model.eval(); pl,ll=[],[]
                with torch.no_grad():
                    A=torch.from_numpy(ADJ).to(device)
                    for b in vl:
                        kps=b['keypoints'].to(device); sc=b['scalars'].to(device)
                        ex_idx=b['exercise_idx'].to(device); lbl=b['label']
                        mode=torch.zeros(kps.shape[0],dtype=torch.long,device=device)
                        lg,_=model(kps,A,sc,mode,ex_idx)
                        pl.extend(lg.argmax(1).cpu().numpy())
                        ll.extend(lbl.numpy())
                model.train()
                from sklearn.metrics import f1_score as _f1
                vf1=_f1(ll,pl,average='macro',zero_division=0)
                print(f"  ep{ep+1:3d} loss={loss:.4f} lr={sch.get_last_lr()[0]:.2e} val_f1={vf1:.3f}")
                if vf1>best_f1:
                    best_f1=vf1; best_state=copy.deepcopy(model.state_dict()); no_imp=0
                else:
                    no_imp+=1
                    if no_imp>=patience:
                        print(f"  Early stop ep{ep+1}"); break

        if best_state: model.load_state_dict(best_state)
        print(f"\n── Eval: {subj} (best val_f1={best_f1:.3f}) ──")
        results.append(_evaluate(model,vl,device))

    accs=[r['acc'] for r in results]; f1s=[r['f1'] for r in results]
    print(f"\n{'='*50}\nLOSO RESULTS\n  Acc={np.mean(accs):.3f}±{np.std(accs):.3f}"
          f"  F1={np.mean(f1s):.3f}±{np.std(f1s):.3f}\n{'='*50}")
    return results, model

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 6 — INFERENCE: sample dict → quality + torque
# ══════════════════════════════════════════════════════════════════════════════

def compute_torque(quality, mode, vel_proxy=0.0):
    m = HARDWARE_MODES.get(mode, HARDWARE_MODES[0])
    torque = (m['bias']-abs(vel_proxy)*0.5) if mode==4 else m['scale']*quality+m['bias']
    return float(np.clip(torque, m['min'], m['max']))

@torch.no_grad()
def infer_sample(model, sample, device='cpu', hardware_mode=1):
    """
    Run one sample dict through trained RehabNet.
    Returns result dict for hardware team.
    """
    model.eval()
    kps_t  = torch.from_numpy(sample['keypoints']).permute(2,0,1).unsqueeze(0).float().to(device)
    sc_t   = torch.from_numpy(sample['scalars']).unsqueeze(0).float().to(device)
    A      = torch.from_numpy(ADJ).to(device)
    mode_t = torch.tensor([hardware_mode], dtype=torch.long, device=device)
    ex_t   = torch.tensor([sample['exercise_idx']], dtype=torch.long, device=device)

    logits, quality, _ = model(kps_t, A, sc_t, mode_t, ex_t)
    prob   = F.softmax(logits, dim=1)[0]
    label  = int(logits.argmax(1).item())
    q      = float(torch.sigmoid(quality).item())
    torque = compute_torque(q, hardware_mode, float(sc_t[0,5]))

    return {
        'rep_id':        sample.get('rep_id','?'),
        'label':         label,            # 0=incorrect, 1=correct
        'correct_prob':  float(prob[1]),
        'quality_score': q,                # 0-1
        'torque_signal': torque,           # 0-1 → hardware
        'mode_name':     HARDWARE_MODES[hardware_mode]['name'],
        'error_flag':    label==0,
    }

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 7 — COMPLETE PATIENT SESSION (video → feedback)
# ══════════════════════════════════════════════════════════════════════════════

def run_patient_video(video_path, patient_id,
                      exercise=None,        # None = auto-classify from first rep
                      model=None, hardware_mode=1, device='cpu'):
    """
    One function to go from raw video to per-rep feedback.
    exercise=None triggers automatic exercise classification from first rep.

    Usage in Colab:
        model = RehabNet()
        model.load_state_dict(torch.load('rehabnet_best.pth', map_location='cpu'))

        results = run_patient_video(
            video_path   = '/content/patient_squat.mp4',
            patient_id   = 'P001',
            exercise     = None,       # auto-classify
            model        = model,
            hardware_mode= 1,
        )
    """
    print(f"\n{'='*50}")
    print(f"Patient: {patient_id}")
    print(f"{'='*50}")

    # PS1: video → DataFrame
    print("\n[PS1] Processing video...")
    df = ps1_process_video(video_path, patient_id, exercise or 'unknown')

    # PS1→PS2 bridge: DataFrame → samples
    print("\n[Bridge] Segmenting reps...")
    samples = ps1_df_to_samples(df, patient_id, exercise or 'unknown')

    if not samples:
        print("No valid reps found — check video quality")
        return []

    # Auto-classify exercise from first rep if not provided
    if exercise is None and model is not None:
        dev     = torch.device(device)
        first   = samples[0]
        kps_t   = torch.from_numpy(first['keypoints']).permute(2,0,1).unsqueeze(0).to(dev)
        adj_t   = torch.from_numpy(ADJ).to(dev)
        pred_ex, pred_idx = model.predict_exercise(kps_t, adj_t)
        exercise = pred_ex
        print(f"\n[Exercise Classification] Predicted: {exercise}")
        # Update all samples with predicted exercise label
        for s in samples:
            s['exercise']     = exercise
            s['exercise_idx'] = pred_idx
    else:
        ex_label = exercise or 'unknown'
        print(f"\n[Exercise] Using: {ex_label}")

    # PS2: RehabNet inference on each rep
    print("\n[PS2] Running RehabNet inference...")
    results = []
    for s in samples:
        out = infer_sample(model, s, device, hardware_mode)
        status = 'CORRECT ✓' if out['label']==1 else 'INCORRECT ✗'
        print(f"  {out['rep_id']:20s} | {status} "
              f"| quality={out['quality_score']:.2f} "
              f"| torque={out['torque_signal']:.2f} "
              f"| mode={out['mode_name']}")
        results.append(out)

    print(f"\nSession complete: {len(results)} reps")
    print(f"  Correct  : {sum(r['label']==1 for r in results)}/{len(results)}")
    print(f"  Avg quality: {np.mean([r['quality_score'] for r in results]):.3f}")
    return results

# ══════════════════════════════════════════════════════════════════════════════
# HOW TO USE IN COLAB
# ══════════════════════════════════════════════════════════════════════════════
"""
─── OPTION A: Train on existing datasets ────────────────────────────────────

from rehab_transformer import load_uiprmd, load_kimore, load_keraal

samples = load_uiprmd() + load_kimore() + load_keraal()
results = run_loso(samples)

─── OPTION B: Process a patient video + run inference ───────────────────────

# Load trained model
model = RehabNet()
model.load_state_dict(torch.load('/content/drive/MyDrive/rehabnet_best.pth'))

# Full pipeline: video → quality + torque
results = run_patient_video(
    video_path    = '/content/patient_video.mp4',
    patient_id    = 'P001',
    exercise      = 'deep_squat',
    model         = model,
    hardware_mode = 1,   # constant_torque
)

─── OPTION C: Use pipeline10 CSV (Keraal already processed) ─────────────────

df = pd.read_csv('/content/KERAAL_ALL_processed.csv')

# group by subject and process each
all_samples = []
for (sid, sess), grp in df.groupby(['subject_id','session']):
    label = 1 if sess=='2' else 0   # professional=1, novice=0
    samples = ps1_df_to_samples(
        grp.reset_index(drop=True),
        patient_id=sid, exercise='ctk_squat', label=label, source='keraal')
    all_samples.extend(samples)

results = run_loso(all_samples)

─── OPTION D: Combine all sources ───────────────────────────────────────────

from rehab_transformer import load_uiprmd, load_kimore
uiprmd_samples  = load_uiprmd()
kimore_samples  = load_kimore()
keraal_samples  = all_samples   # from Option C above

all_samples = uiprmd_samples + kimore_samples + keraal_samples
results     = run_loso(all_samples)
"""

"\n─── OPTION A: Train on existing datasets ────────────────────────────────────\n\nfrom rehab_transformer import load_uiprmd, load_kimore, load_keraal\n\nsamples = load_uiprmd() + load_kimore() + load_keraal()\nresults = run_loso(samples)\n\n─── OPTION B: Process a patient video + run inference ───────────────────────\n\n# Load trained model\nmodel = RehabNet()\nmodel.load_state_dict(torch.load('/content/drive/MyDrive/rehabnet_best.pth'))\n\n# Full pipeline: video → quality + torque\nresults = run_patient_video(\n    video_path    = '/content/patient_video.mp4',\n    patient_id    = 'P001',\n    exercise      = 'deep_squat',\n    model         = model,\n    hardware_mode = 1,   # constant_torque\n)\n\n─── OPTION C: Use pipeline10 CSV (Keraal already processed) ─────────────────\n\ndf = pd.read_csv('/content/KERAAL_ALL_processed.csv')\n\n# group by subject and process each\nall_samples = []\nfor (sid, sess), grp in df.groupby(['subject_id','session']):\n    label = 1 if sess=='2' else 

In [6]:
"""
PS3 Hardware Integration
Replaces the mock InferenceEngine from integratepipeline.ipynb
with real RehabNet inference connected to hardware torque output.

Hardware modes:
    0 = assist  (patient/injured — device helps)
    1 = resist  (healthy/rehab-complete — device resists)
"""

import os, json, pickle, logging
from datetime import datetime
import numpy as np
import torch

# ── Logging (from integratepipeline.ipynb) ────────────────────────────────────

LOG_PATH = "pipeline.log"
logger = logging.getLogger("rehab_pipeline")
logger.setLevel(logging.DEBUG)

if not logger.handlers:
    fh = logging.FileHandler(LOG_PATH)
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(logging.Formatter(
        "%(asctime)s | %(levelname)-8s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"))
    logger.addHandler(fh)
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    ch.setFormatter(logging.Formatter("%(levelname)-8s | %(message)s"))
    logger.addHandler(ch)

def log(level, msg):
    getattr(logger, level.lower())(msg)

# ── PS3 InferenceEngine — real implementation ─────────────────────────────────

class PS3InferenceEngine:
    """
    Real InferenceEngine replacing integratepipeline.ipynb mock.

    Connects RehabNet predictions to hardware torque output.
    Input:  per-rep result dict from run_patient_video()
    Output: hardware command dict with torque signal

    Hardware interface (stub — replace with actual serial/BLE calls):
        send_torque_command(torque: float, mode: int) -> None
    """

    def __init__(self, model=None, device='cpu', hardware_mode=0):
        self.model        = model
        self.device       = torch.device(device)
        self.hardware_mode = hardware_mode  # 0=assist, 1=resist
        self.session_log  = []
        self.connected    = False
        log("info", f"PS3InferenceEngine init — mode={HARDWARE_MODES[hardware_mode]['name']}")

    def load_model(self, model_path: str):
        """Load trained RehabNet weights."""
        m = RehabNet().to(self.device)
        m.load_state_dict(torch.load(model_path, map_location=self.device))
        m.eval()
        self.model = m
        log("info", f"Model loaded: {model_path}")
        print(f"  Model loaded from {model_path}")

    def connect_hardware(self, port=None):
        """
        Connect to rehabilitation device.
        Replace with actual serial/BLE connection:
            import serial
            self.ser = serial.Serial(port, baudrate=115200)
        """
        self.connected = True
        log("info", f"Hardware connected (stub) — port={port}")
        print(f"  Hardware connected (stub mode — replace with real serial/BLE)")

    def send_torque_command(self, torque: float, mode: int):
        """
        Send torque command to device.
        Replace stub with actual hardware write:
            cmd = f"TORQUE:{torque:.3f}\n"
            self.ser.write(cmd.encode())
        """
        log("debug", f"TORQUE CMD: {torque:.3f}  mode={HARDWARE_MODES[mode]['name']}")
        # print(f"  [HW] torque={torque:.3f}  mode={mode}")  # uncomment for debug

    def predict(self, rep_result: dict) -> dict:
        """
        Convert a per-rep result dict from run_patient_video() into
        a hardware command + structured prediction.

        rep_result keys: label, quality_score, torque_signal, mode_name, etc.
        """
        torque = rep_result.get('torque_signal', 0.5)
        mode   = rep_result.get('mode', self.hardware_mode)
        label  = rep_result.get('label', 1)
        q      = rep_result.get('quality_score', 0.5)

        # Send to hardware
        if self.connected:
            self.send_torque_command(torque, mode)

        command = {
            'timestamp':      datetime.now().isoformat(),
            'torque':         round(torque, 4),
            'mode':           mode,
            'mode_name':      HARDWARE_MODES[mode]['name'],
            'quality_score':  round(q, 4),
            'label':          label,
            'correct':        label == 1,
            'patient_feedback': rep_result.get('feedback', ''),
        }

        self.session_log.append(command)
        return command

    def run_session(self, video_path: str, patient_id: str,
                    exercise: str = None, hardware_mode: int = None) -> dict:
        """
        Full session: video → PS1 → PS2 → PS3 hardware commands.
        This is the main entry point for a clinical session.
        """
        if hardware_mode is not None:
            self.hardware_mode = hardware_mode

        log("info", f"Session start: {patient_id}  video={video_path}")

        # Run full PS1+PS2 pipeline
        rep_results = run_patient_video(
            video_path    = video_path,
            patient_id    = patient_id,
            exercise      = exercise,
            model         = self.model,
            hardware_mode = self.hardware_mode,
            device        = str(self.device),
        )

        if not rep_results:
            log("warning", "No reps found in video")
            return {'error': 'No reps found', 'patient_id': patient_id}

        # Generate hardware commands for each rep
        commands = [self.predict(r) for r in rep_results]

        # Session summary
        n_correct = sum(r['label']==1 for r in rep_results)
        avg_q     = np.mean([r['quality_score'] for r in rep_results])
        avg_t     = np.mean([c['torque']        for c in commands])

        summary = {
            'patient_id':      patient_id,
            'exercise':        rep_results[0].get('exercise', exercise or 'unknown'),
            'n_reps':          len(rep_results),
            'n_correct':       n_correct,
            'pct_correct':     round(n_correct / len(rep_results) * 100, 1),
            'avg_quality':     round(float(avg_q), 3),
            'avg_torque':      round(float(avg_t), 3),
            'hardware_mode':   HARDWARE_MODES[self.hardware_mode]['name'],
            'hardware_commands': commands,
            'timestamp':       datetime.now().isoformat(),
        }

        log("info", f"Session complete: {len(rep_results)} reps  "
                    f"correct={n_correct}/{len(rep_results)}  "
                    f"avg_quality={avg_q:.3f}")

        return summary

    def save_session_report(self, summary: dict, save_dir: str = 'reports'):
        """Save session report to JSON — mirrors integratepipeline report structure."""
        os.makedirs(save_dir, exist_ok=True)
        fname = f"{summary['patient_id']}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        path  = os.path.join(save_dir, fname)
        with open(path, 'w') as f:
            json.dump(summary, f, indent=2, default=str)
        log("info", f"Session report saved: {path}")
        print(f"  Report saved: {path}")
        return path


# ── Pipeline orchestration (from integratepipeline.ipynb) ────────────────────

def run_complete_pipeline(
    video_path:       str,
    patient_id:       str,
    model_path:       str = 'rehabnet_best.pth',
    exercise_clf_path: str = 'exercise_classifier.pkl',
    hardware_mode:    int = 0,      # 0=assist, 1=resist
    hardware_port:    str = None,   # serial port — None = stub mode
    save_report:      bool = True,
    device:           str = 'cuda' if torch.cuda.is_available() else 'cpu',
) -> dict:
    """
    Master pipeline — one function to run everything.

    PS1 (MediaPipe) → PS2 (RehabNet) → PS3 (hardware torque)

    Args:
        video_path:        path to patient video
        patient_id:        patient identifier
        model_path:        path to rehabnet_best.pth
        exercise_clf_path: path to exercise_classifier.pkl
        hardware_mode:     0=assist (patient), 1=resist (healthy)
        hardware_port:     serial port string or None for stub
        save_report:       save JSON report to reports/
        device:            'cuda' or 'cpu'

    Returns:
        session summary dict with per-rep results and hardware commands
    """
    log("info", "=" * 55)
    log("info", "PIPELINE START")
    log("info", f"  patient={patient_id}  video={os.path.basename(video_path)}")
    log("info", f"  mode={HARDWARE_MODES[hardware_mode]['name']}  device={device}")

    # Step 1: Load exercise classifier
    load_exercise_classifier(exercise_clf_path)

    # Step 2: Set up PS3 engine with trained model
    engine = PS3InferenceEngine(device=device, hardware_mode=hardware_mode)
    engine.load_model(model_path)

    # Step 3: Connect hardware
    engine.connect_hardware(port=hardware_port)

    # Step 4: Run full session
    summary = engine.run_session(
        video_path    = video_path,
        patient_id    = patient_id,
        exercise      = None,   # auto-classify
        hardware_mode = hardware_mode,
    )

    # Step 5: Save report
    if save_report and 'error' not in summary:
        engine.save_session_report(summary)

    # Step 6: Print summary
    print(f"\n{'='*50}")
    print(f"PIPELINE COMPLETE")
    print(f"{'='*50}")
    print(f"  Patient:      {summary.get('patient_id')}")
    print(f"  Exercise:     {summary.get('exercise')}")
    print(f"  Reps:         {summary.get('n_reps')}")
    print(f"  Correct:      {summary.get('n_correct')}/{summary.get('n_reps')} "
          f"({summary.get('pct_correct')}%)")
    print(f"  Avg quality:  {summary.get('avg_quality')}")
    print(f"  Avg torque:   {summary.get('avg_torque')}")
    print(f"  HW mode:      {summary.get('hardware_mode')}")

    log("info", "PIPELINE COMPLETE")
    return summary


print("PS3 Hardware Integration ready.")
print("Modes: 0=assist (patient/injured)  1=resist (healthy)")


PS3 Hardware Integration ready.
Modes: 0=assist (patient/injured)  1=resist (healthy)


In [11]:
import inspect
src = inspect.getsource(RehabNet.__init__)
print('bilstm in code:', 'bilstm' in src)
print('exercise_classify_head in code:', 'exercise_classify_head' in src)

bilstm in code: True
exercise_classify_head in code: False


In [15]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 5 — paste this into your Colab Cell 5 (replaces existing Cell 5)
#
# Changes from old Cell 5:
#   - run_loso() called correctly (no args needed — builds dataset internally)
#   - returns (results, trained_model) tuple
#   - saves model to Drive automatically inside run_loso()
# ═══════════════════════════════════════════════════════════════════════════

from google.colab import drive
drive.mount('/content/drive')

import torch, os
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# Train RehabNet
results, trained_model = run_loso(n_epochs=60, batch_size=32, lr=3e-4)

# Verify saved weights
keys = list(torch.load('/content/drive/MyDrive/models/rehabnet_best.pth',
                        map_location='cpu').keys())
print('bilstm saved:          ', any('bilstm'             in k for k in keys))
print('exercise head saved:   ', any('ex_head'            in k for k in keys))
print('classify head saved:   ', any('classify_head'      in k for k in keys))
print('Best F1:', round(max(r['f1'] for r in results), 3))

Mounted at /content/drive
Device: cuda
Device: cuda
UI-PRMD: 1180 samples
KIMORE: 77 samples
Keraal: 301 samples

Master: 1558 samples
  correct=722  incorrect=836
  sources:   {'kimore', 'keraal', 'uiprmd'}
  exercises: {'side_lunge', 'deep_squat', 'sit_to_stand', 'straight_leg_raise', 'ctk_squat', 'inline_lunge', 'hurdle_step', 'squat'}

LOSO held-out: uiprmd_s01
  class weights: incorrect=0.927  correct=1.086
  ep  5 loss=0.8147 lr=3.00e-04 val_f1=0.566
  ep 10 loss=0.6594 lr=2.94e-04 val_f1=0.715
  ep 15 loss=0.6093 lr=2.76e-04 val_f1=0.722
  ep 20 loss=0.5462 lr=2.48e-04 val_f1=0.751
  ep 25 loss=0.5333 lr=2.12e-04 val_f1=0.733
  ep 30 loss=0.4510 lr=1.71e-04 val_f1=0.747
  ep 35 loss=0.4059 lr=1.29e-04 val_f1=0.708
  ep 40 loss=0.3957 lr=8.77e-05 val_f1=0.766
  ep 45 loss=0.3798 lr=5.18e-05 val_f1=0.767
  ep 50 loss=0.3374 lr=2.38e-05 val_f1=0.758
  ep 55 loss=0.3483 lr=6.08e-06 val_f1=0.767
  ep 60 loss=0.3540 lr=0.00e+00 val_f1=0.775

── Eval: uiprmd_s01 (best val_f1=0.775) ──


In [17]:
# ── Full pipeline: video → PS1 → PS2 → PS3 hardware ─────────────
summary = run_complete_pipeline(
    video_path        = '/content/8837221-uhd_2160_4096_25fps.mp4',
    patient_id        = 'P001',
    model_path        = '/content/drive/MyDrive/models/rehabnet_best.pth',
    exercise_clf_path = 'exercise_classifier.pkl',
    hardware_mode     = 0,   # 0=assist (patient), 1=resist (healthy)
    hardware_port     = None,  # None=stub; set to '/dev/ttyUSB0' for real device
    save_report       = True,
)

# ── Per-rep details ───────────────────────────────────────────────
print('\nPer-rep breakdown:')
for i, cmd in enumerate(summary.get('hardware_commands', [])):
    print(f"  Rep {i+1:2d}: {'✓' if cmd['correct'] else '✗'} "
          f"q={cmd['quality_score']:.2f}  "
          f"torque={cmd['torque']:.3f}  "
          f"mode={cmd['mode_name']}")

INFO     | =======================================================
INFO:rehab_pipeline:=======================================================
INFO     | PIPELINE START
INFO:rehab_pipeline:PIPELINE START
INFO     |   patient=P001  video=8837221-uhd_2160_4096_25fps.mp4
INFO:rehab_pipeline:  patient=P001  video=8837221-uhd_2160_4096_25fps.mp4
INFO     |   mode=assist  device=cuda
INFO:rehab_pipeline:  mode=assist  device=cuda
INFO     | PS3InferenceEngine init — mode=assist
INFO:rehab_pipeline:PS3InferenceEngine init — mode=assist


No saved classifier at exercise_classifier.pkl — will use neural head fallback


INFO     | Model loaded: /content/drive/MyDrive/models/rehabnet_best.pth
INFO:rehab_pipeline:Model loaded: /content/drive/MyDrive/models/rehabnet_best.pth
INFO     | Hardware connected (stub) — port=None
INFO:rehab_pipeline:Hardware connected (stub) — port=None
INFO     | Session start: P001  video=/content/8837221-uhd_2160_4096_25fps.mp4
INFO:rehab_pipeline:Session start: P001  video=/content/8837221-uhd_2160_4096_25fps.mp4


  Model loaded from /content/drive/MyDrive/models/rehabnet_best.pth
  Hardware connected (stub mode — replace with real serial/BLE)

Patient: P001

[PS1] Processing video...
  Downloaded.
  PS1: 639 frames  detection=100%  79.0s

[Bridge] Segmenting reps...
  Bridge: 6 reps found → 6 valid samples

[Exercise Classification] Predicted: inline_lunge

[PS2] Running RehabNet inference...
  P001_r1              | CORRECT ✓ | quality=0.90 | torque=0.23 | mode=assist
  P001_r2              | CORRECT ✓ | quality=0.61 | torque=0.32 | mode=assist
  P001_r3              | CORRECT ✓ | quality=0.69 | torque=0.29 | mode=assist
  P001_r4              | INCORRECT ✗ | quality=0.03 | torque=0.49 | mode=assist
  P001_r5              | CORRECT ✓ | quality=0.75 | torque=0.28 | mode=assist


DEBUG:rehab_pipeline:TORQUE CMD: 0.231  mode=assist
DEBUG:rehab_pipeline:TORQUE CMD: 0.318  mode=assist
DEBUG:rehab_pipeline:TORQUE CMD: 0.292  mode=assist
DEBUG:rehab_pipeline:TORQUE CMD: 0.492  mode=assist
DEBUG:rehab_pipeline:TORQUE CMD: 0.276  mode=assist
DEBUG:rehab_pipeline:TORQUE CMD: 0.283  mode=assist
INFO     | Session complete: 6 reps  correct=5/6  avg_quality=0.616
INFO:rehab_pipeline:Session complete: 6 reps  correct=5/6  avg_quality=0.616
INFO     | Session report saved: reports/P001_20260611_073506.json
INFO:rehab_pipeline:Session report saved: reports/P001_20260611_073506.json
INFO     | PIPELINE COMPLETE
INFO:rehab_pipeline:PIPELINE COMPLETE


  P001_r6              | CORRECT ✓ | quality=0.72 | torque=0.28 | mode=assist

Session complete: 6 reps
  Correct  : 5/6
  Avg quality: 0.616
  Report saved: reports/P001_20260611_073506.json

PIPELINE COMPLETE
  Patient:      P001
  Exercise:     unknown
  Reps:         6
  Correct:      5/6 (83.3%)
  Avg quality:  0.616
  Avg torque:   0.315
  HW mode:      assist

Per-rep breakdown:
  Rep  1: ✓ q=0.90  torque=0.231  mode=assist
  Rep  2: ✓ q=0.61  torque=0.318  mode=assist
  Rep  3: ✓ q=0.69  torque=0.292  mode=assist
  Rep  4: ✗ q=0.03  torque=0.492  mode=assist
  Rep  5: ✓ q=0.75  torque=0.276  mode=assist
  Rep  6: ✓ q=0.72  torque=0.283  mode=assist
